### Preprocess Mooring Data 
This notebook is for preprocessing mooring data. This includes correcting velocity for magnetic delcination, despiking data, and renaming variables to be consistent in the data tree 

Useful links for outlier detection methods: 

- https://medium.com/@alex.eslava96/outlier-detection-techniques-for-time-series-9868db2875c2

- https://medium.com/@tubelwj/python-outlier-detection-iqr-method-and-z-score-implementation-8e825edf4b32

- https://towardsdatascience.com/the-ultimate-guide-to-finding-outliers-in-your-time-series-data-part-1-1bf81e09ade4-2/

- https://towardsdatascience.com/how-to-deal-with-time-series-outliers-28b217c7f6c2/



In [ ]:
# import packages 
import xarray as xr 
import numpy as np 
import matplotlib.pyplot as plt 
import pandas as pd 
import glob
from tqdm import tqdm
from scipy import stats 
from scipy.stats import median_abs_deviation
import sys
sys.path.append('/Users/mblan/Documents/research/TNB_tools/TNB_tools')
import magnetic_declination

In [ ]:
%matplotlib widget

In [ ]:
def summarize_all(dt):
    """Return a DataFrame summarizing the entire DataTree."""
    import pandas as pd
    import numpy as np

    rows = []

    for mooring, mooring_node in dt.children.items():
        for year, year_node in mooring_node.children.items():
            for instrument, instr_node in year_node.children.items():
                for position, pos_node in instr_node.children.items():

                    ds = pos_node.ds

                    if "time" in ds.coords:
                        t = ds["time"].to_numpy()
                        t_start = t[0]
                        t_end = t[-1]
                        dt_med = np.median(np.diff(t)) if len(t) > 1 else np.nan
                    else:
                        t_start = t_end = dt_med = np.nan

                    depth = ds.coords["z"].item() if "z" in ds.coords else np.nan

                    rows.append({
                        "mooring": mooring,
                        "year": year,
                        "instrument": instrument,
                        "position": position,
                        "depth": depth,
                        "variables": list(ds.data_vars),
                        "time_start": t_start,
                        "time_end": t_end,
                        "sampling_period": dt_med,
                        "n_points": ds.sizes.get("time", np.nan),
                    })

    df = pd.DataFrame(rows)
    df = df.sort_values(["mooring", "year", "instrument", "position"])
    return df.reset_index(drop=True)


In [ ]:
def plot_timeseries(years,var,mooring,instrument,position,color,label):
    """
    inputs:
    years: years to loop over (array containing years in string format)
    var: variable of interest (string)
    mooring: mooring of interest (string)
    instrument: instrument of interest (string)
    position: position of interest (string)
    color: color to use for line plotting (string)
    """
    for i in range(len(years)):
        var = var
        ds = dt[mooring][years[i]][instrument][position]
        if var in ds.data_vars:
            ds[var].plot(color=color,label=label)

### Load Mooring D data 

In [ ]:
# loop through all folders and print paths for netcdf files 
folders = glob.glob("//thepenguin/penguin2/Data/TNB/mooring/Mooring_D_MORSea/Mooring_D_2014_2025/Mooring_D/*")
files = []
for folder in tqdm(folders):
    print(folder)
    files.append(glob.glob(folder+'/*.nc'))

In [ ]:
# remove first entry which is not a folder with data 
mooring_d_files = files[:-1]

In [ ]:
mooring_d_files[0]

In [ ]:
D_2014_vel_top = xr.open_dataset(mooring_d_files[0][0])
D_2014_vel_mid = xr.open_dataset(mooring_d_files[0][1])
D_2014_vel_bot = xr.open_dataset(mooring_d_files[0][6])
D_2014_ctd_top = xr.open_dataset(mooring_d_files[0][2])
D_2014_temp_mid1 = xr.open_dataset(mooring_d_files[0][5])
D_2014_temp_mid2 = xr.open_dataset(mooring_d_files[0][4])
D_2014_ctd_bot = xr.open_dataset(mooring_d_files[0][3])

In [ ]:
# drop depth from dimension and add lat/lon to coordinates 
D_2014_vel_top = D_2014_vel_top.rename({'depth':'z'}).squeeze().assign_coords({'lat':-75.13573,'lon':164.551})
D_2014_vel_mid = D_2014_vel_mid.rename({'depth':'z'}).squeeze().assign_coords({'lat':-75.13573,'lon':164.551})
D_2014_vel_bot = D_2014_vel_bot.rename({'depth':'z'}).squeeze().assign_coords({'lat':-75.13573,'lon':164.551})
D_2014_ctd_top = D_2014_ctd_top.rename({'depth':'z'}).squeeze().assign_coords({'lat':-75.13573,'lon':164.551})
D_2014_temp_mid1 = D_2014_temp_mid1.rename({'depth':'z'}).squeeze().assign_coords({'lat':-75.13573,'lon':164.551})
D_2014_temp_mid2 = D_2014_temp_mid2.rename({'depth':'z'}).squeeze().assign_coords({'lat':-75.13573,'lon':164.551})
D_2014_ctd_bot = D_2014_ctd_bot.rename({'depth':'z'}).squeeze().assign_coords({'lat':-75.13573,'lon':164.551})

In [ ]:
mooring_d_files[1]

In [ ]:
D_2016_vel_top = xr.open_dataset(mooring_d_files[1][0])
D_2016_vel_mid = xr.open_dataset(mooring_d_files[1][1])
D_2016_vel_bot = xr.open_dataset(mooring_d_files[1][2])
D_2016_ctd_top = xr.open_dataset(mooring_d_files[1][3])
D_2016_temp_mid1 = xr.open_dataset(mooring_d_files[1][6])
D_2016_temp_mid2 = xr.open_dataset(mooring_d_files[1][5])
D_2016_ctd_bot = xr.open_dataset(mooring_d_files[1][4])

In [ ]:
# drop depth from dimension and add lat/lon to coordinates 
D_2016_vel_top = D_2016_vel_top.rename({'depth':'z'}).squeeze().assign_coords({'lat':-75.13587,'lon':164.5549})
D_2016_vel_mid = D_2016_vel_mid.rename({'depth':'z'}).squeeze().assign_coords({'lat':-75.13587,'lon':164.5549})
D_2016_vel_bot = D_2016_vel_bot.rename({'depth':'z'}).squeeze().assign_coords({'lat':-75.13587,'lon':164.5549})
D_2016_ctd_top = D_2016_ctd_top.rename({'depth':'z'}).squeeze().assign_coords({'lat':-75.13587,'lon':164.5549})
D_2016_temp_mid1 = D_2016_temp_mid1.rename({'depth':'z'}).squeeze().assign_coords({'lat':-75.13587,'lon':164.5549})
D_2016_temp_mid2 = D_2016_temp_mid2.rename({'depth':'z'}).squeeze().assign_coords({'lat':-75.13587,'lon':164.5549})
D_2016_ctd_bot = D_2016_ctd_bot.rename({'depth':'z'}).squeeze().assign_coords({'lat':-75.13587,'lon':164.5549})

In [ ]:
mooring_d_files[2]

In [ ]:
D_2017_vel_top = xr.open_dataset(mooring_d_files[2][0])
D_2017_vel_mid = xr.open_dataset(mooring_d_files[2][5])
D_2017_vel_bot = xr.open_dataset(mooring_d_files[2][6])
D_2017_ctd_top = xr.open_dataset(mooring_d_files[2][1])
D_2017_ctd_mid1 = xr.open_dataset(mooring_d_files[2][3])
D_2017_temp_mid2 = xr.open_dataset(mooring_d_files[2][4])
D_2017_ctd_bot = xr.open_dataset(mooring_d_files[2][2])

In [ ]:
D_2017_vel_top = D_2017_vel_top.rename({'depth':'z'}).squeeze().assign_coords({'lat':-75.13611,'lon':164.58958})
D_2017_vel_mid = D_2017_vel_mid.rename({'depth':'z'}).squeeze().assign_coords({'lat':-75.13611,'lon':164.58958})
D_2017_vel_bot = D_2017_vel_bot.rename({'depth':'z'}).squeeze().assign_coords({'lat':-75.13611,'lon':164.58958})
D_2017_ctd_top = D_2017_ctd_top.rename({'depth':'z'}).squeeze().assign_coords({'lat':-75.13611,'lon':164.58958})
D_2017_ctd_mid1 = D_2017_ctd_mid1.rename({'depth':'z'}).squeeze().assign_coords({'lat':-75.13611,'lon':164.58958})
D_2017_temp_mid2 = D_2017_temp_mid2.rename({'depth':'z'}).squeeze().assign_coords({'lat':-75.13611,'lon':164.58958})
D_2017_ctd_bot = D_2017_ctd_bot.rename({'depth':'z'}).squeeze().assign_coords({'lat':-75.13611,'lon':164.58958})

In [ ]:
mooring_d_files[3]

In [ ]:
D_2019_vel_top = xr.open_dataset(mooring_d_files[3][0])
D_2019_vel_mid = xr.open_dataset(mooring_d_files[3][5])
D_2019_vel_bot = xr.open_dataset(mooring_d_files[3][6])
D_2019_ctd_top = xr.open_dataset(mooring_d_files[3][1])
D_2019_ctd_mid1 = xr.open_dataset(mooring_d_files[3][2])
D_2019_temp_mid2 = xr.open_dataset(mooring_d_files[3][4])
D_2019_ctd_bot = xr.open_dataset(mooring_d_files[3][3])

In [ ]:
D_2019_vel_top = D_2019_vel_top.rename({'depth':'z'}).squeeze().assign_coords({'lat':-75.07661,'lon':164.72989})
D_2019_vel_mid = D_2019_vel_mid.rename({'depth':'z'}).squeeze().assign_coords({'lat':-75.07661,'lon':164.72989})
D_2019_vel_bot = D_2019_vel_bot.rename({'depth':'z'}).squeeze().assign_coords({'lat':-75.07661,'lon':164.72989})
D_2019_ctd_top = D_2019_ctd_top.rename({'depth':'z'}).squeeze().assign_coords({'lat':-75.07661,'lon':164.72989})
D_2019_ctd_mid1 = D_2019_ctd_mid1.rename({'depth':'z'}).squeeze().assign_coords({'lat':-75.07661,'lon':164.72989})
D_2019_temp_mid2 = D_2019_temp_mid2.rename({'depth':'z'}).squeeze().assign_coords({'lat':-75.07661,'lon':164.72989})
D_2019_ctd_bot = D_2019_ctd_bot.rename({'depth':'z'}).squeeze().assign_coords({'lat':-75.07661,'lon':164.72989})

In [ ]:
mooring_d_files[4]

In [ ]:
D_2020_vel_top = xr.open_dataset(mooring_d_files[4][4])
D_2020_vel_mid = xr.open_dataset(mooring_d_files[4][3])
# D_2020_2021_vel_bot = xr.open_dataset(mooring_d_files[4][3])
D_2020_ctd_top = xr.open_dataset(mooring_d_files[4][0])
D_2020_temp_mid1 = xr.open_dataset(mooring_d_files[4][2])
D_2020_ctd_mid2 = xr.open_dataset(mooring_d_files[4][1])
# D_2020_2021_sbe_bot = xr.open_dataset(mooring_d_files[4][1])

In [ ]:
D_2020_vel_top = D_2020_vel_top.rename({'depth':'z'}).squeeze().assign_coords({'lat':-75.1237,'lon':164.95062})
D_2020_vel_mid = D_2020_vel_mid.rename({'depth':'z'}).squeeze().assign_coords({'lat':-75.1237,'lon':164.95062})
D_2020_ctd_top = D_2020_ctd_top.rename({'depth':'z'}).squeeze().assign_coords({'lat':-75.1237,'lon':164.95062})
D_2020_temp_mid1 = D_2020_temp_mid1.rename({'depth':'z'}).squeeze().assign_coords({'lat':-75.1237,'lon':164.95062})
D_2020_ctd_mid2 = D_2020_ctd_mid2.rename({'depth':'z'}).squeeze().assign_coords({'lat':-75.1237,'lon':164.95062})

In [ ]:
mooring_d_files[5]

In [ ]:
D_2021_vel_top = xr.open_dataset(mooring_d_files[5][6])
D_2021_vel_mid = xr.open_dataset(mooring_d_files[5][5])
D_2021_vel_bot = xr.open_dataset(mooring_d_files[5][4])
D_2021_ctd_top = xr.open_dataset(mooring_d_files[5][0])
D_2021_ctd_mid1 = xr.open_dataset(mooring_d_files[5][1])
D_2021_temp_mid2 = xr.open_dataset(mooring_d_files[5][3])
D_2021_ctd_bot = xr.open_dataset(mooring_d_files[5][2])

In [ ]:
D_2021_vel_top = D_2021_vel_top.rename({'depth':'z'}).squeeze().assign_coords({'lat':-75.1237,'lon':164.95062})
D_2021_vel_mid = D_2021_vel_mid.rename({'depth':'z'}).squeeze().assign_coords({'lat':-75.1237,'lon':164.95062})
D_2021_vel_bot = D_2021_vel_bot.rename({'depth':'z'}).squeeze().assign_coords({'lat':-75.1237,'lon':164.95062})
D_2021_ctd_top = D_2021_ctd_top.rename({'depth':'z'}).squeeze().assign_coords({'lat':-75.1237,'lon':164.95062})
D_2021_ctd_mid1 = D_2021_ctd_mid1.rename({'depth':'z'}).squeeze().assign_coords({'lat':-75.1237,'lon':164.95062})
D_2021_temp_mid2 = D_2021_temp_mid2.rename({'depth':'z'}).squeeze().assign_coords({'lat':-75.1237,'lon':164.95062})
D_2021_ctd_bot = D_2021_ctd_bot.rename({'depth':'z'}).squeeze().assign_coords({'lat':-75.1237,'lon':164.95062})

In [ ]:
mooring_d_files[6]

In [ ]:
D_2022_vel_top = xr.open_dataset(mooring_d_files[6][3])
D_2022_vel_mid = xr.open_dataset(mooring_d_files[6][4])
# D_2022_2023_vel_bot = xr.open_dataset(mooring_d_files[6][4])
D_2022_ctd_top = xr.open_dataset(mooring_d_files[6][1])
D_2022_ctd_mid1 = xr.open_dataset(mooring_d_files[6][0])
D_2022_temp_mid2 = xr.open_dataset(mooring_d_files[6][2])
# D_2022_2023_sbe_bot = xr.open_dataset(mooring_d_files[6][2])

In [ ]:
D_2022_vel_top = D_2022_vel_top.rename({'depth':'z'}).squeeze().assign_coords({'lat':-75.1274,'lon':164.9707})
D_2022_vel_mid = D_2022_vel_mid.rename({'depth':'z'}).squeeze().assign_coords({'lat':-75.1274,'lon':164.9707})
D_2022_ctd_top = D_2022_ctd_top.rename({'depth':'z'}).squeeze().assign_coords({'lat':-75.1274,'lon':164.9707})
D_2022_ctd_mid1 = D_2022_ctd_mid1.rename({'depth':'z'}).squeeze().assign_coords({'lat':-75.1274,'lon':164.9707})
D_2022_temp_mid2 = D_2022_temp_mid2.rename({'depth':'z'}).squeeze().assign_coords({'lat':-75.1274,'lon':164.9707})

In [ ]:
mooring_d_files[7]

In [ ]:
# D_2023_2024_vel_top = xr.open_dataset(mooring_d_files[6][3])
D_2023_vel_mid = xr.open_dataset(mooring_d_files[7][3])
D_2023_vel_bot = xr.open_dataset(mooring_d_files[7][4])
# D_2023_2024_sbe_top = xr.open_dataset(mooring_d_files[6][1])
D_2023_ctd_mid1 = xr.open_dataset(mooring_d_files[7][1])
D_2023_temp_mid2 = xr.open_dataset(mooring_d_files[7][2])
D_2023_ctd_bot = xr.open_dataset(mooring_d_files[7][0])

In [ ]:
D_2023_vel_mid = D_2023_vel_mid.rename({'depth':'z'}).squeeze().assign_coords({'lat':-75.1667,'lon':164.5976})
D_2023_vel_bot = D_2023_vel_bot.rename({'depth':'z'}).squeeze().assign_coords({'lat':-75.1667,'lon':164.5976})
D_2023_ctd_mid1 = D_2023_ctd_mid1.rename({'depth':'z'}).squeeze().assign_coords({'lat':-75.1667,'lon':164.5976})
D_2023_temp_mid2 = D_2023_temp_mid2.rename({'depth':'z'}).squeeze().assign_coords({'lat':-75.1667,'lon':164.5976})
D_2023_ctd_bot = D_2023_ctd_bot.rename({'depth':'z'}).squeeze().assign_coords({'lat':-75.1667,'lon':164.5976})

In [ ]:
mooring_d_files[8]

In [ ]:
# D_2024_2025_vel_top = xr.open_dataset(mooring_d_files[8][3])
D_2024_vel_mid = xr.open_dataset(mooring_d_files[8][3])
D_2024_vel_bot = xr.open_dataset(mooring_d_files[8][4])
# D_2024_2025_sbe_top = xr.open_dataset(mooring_d_files[8][1])
D_2024_ctd_mid1 = xr.open_dataset(mooring_d_files[8][1])
D_2024_temp_mid2 = xr.open_dataset(mooring_d_files[8][2])
D_2024_ctd_bot = xr.open_dataset(mooring_d_files[8][0])

In [ ]:
D_2024_vel_mid = D_2024_vel_mid.rename({'depth':'z'}).squeeze().assign_coords({'lat':-75.166711,'lon':164.597667})
D_2024_vel_bot = D_2024_vel_bot.rename({'depth':'z'}).squeeze().assign_coords({'lat':-75.166711,'lon':164.597667})
D_2024_ctd_mid1 = D_2024_ctd_mid1.rename({'depth':'z'}).squeeze().assign_coords({'lat':-75.166711,'lon':164.597667})
D_2024_temp_mid2 = D_2024_temp_mid2.rename({'depth':'z'}).squeeze().assign_coords({'lat':-75.166711,'lon':164.597667})
D_2024_ctd_bot = D_2024_ctd_bot.rename({'depth':'z'}).squeeze().assign_coords({'lat':-75.166711,'lon':164.597667})

In [ ]:
# loop through and add all dataset to the datatree 
# make list of all datasets to loop through 
D_datasets = [
    D_2014_vel_top,
    D_2014_vel_mid,
    D_2014_vel_bot,
    D_2014_ctd_top,
    D_2014_temp_mid1,
    D_2014_temp_mid2,
    D_2014_ctd_bot,
    D_2016_vel_top,
    D_2016_vel_mid,
    D_2016_vel_bot,
    D_2016_ctd_top,
    D_2016_temp_mid1,
    D_2016_temp_mid2,
    D_2016_ctd_bot,
    D_2017_vel_top,
    D_2017_vel_mid,
    D_2017_vel_bot,
    D_2017_ctd_top,
    D_2017_ctd_mid1,
    D_2017_temp_mid2,
    D_2017_ctd_bot,
    D_2019_vel_top,
    D_2019_vel_mid,
    D_2019_vel_bot,
    D_2019_ctd_top,
    D_2019_ctd_mid1,
    D_2019_temp_mid2,
    D_2019_ctd_bot,
    D_2020_vel_top,
    D_2020_vel_mid,
    D_2020_ctd_top,
    D_2020_temp_mid1,
    D_2020_ctd_mid2,
    D_2021_vel_top,
    D_2021_vel_mid,
    D_2021_vel_bot,
    D_2021_ctd_top,
    D_2021_ctd_mid1,
    D_2021_temp_mid2,
    D_2021_ctd_bot,
    D_2022_vel_top,
    D_2022_vel_mid,
    D_2022_ctd_top,
    D_2022_ctd_mid1,
    D_2022_temp_mid2,
    D_2023_vel_mid,
    D_2023_vel_bot,
    D_2023_ctd_mid1,
    D_2023_temp_mid2,
    D_2023_ctd_bot,
    D_2024_vel_mid,
    D_2024_vel_bot,
    D_2024_ctd_mid1,
    D_2024_temp_mid2,
    D_2024_ctd_bot
]

In [ ]:
# loop through and add all dataset to the datatree 
# make list of all datasets to loop through 
dataset_names = [
    'D_2014_vel_top',
    'D_2014_vel_mid',
    'D_2014_vel_bot',
    'D_2014_ctd_top',
    'D_2014_temp_mid1',
    'D_2014_temp_mid2',
    'D_2014_ctd_bot',
    'D_2016_vel_top',
    'D_2016_vel_mid',
    'D_2016_vel_bot',
    'D_2016_ctd_top',
    'D_2016_temp_mid1',
    'D_2016_temp_mid2',
    'D_2016_ctd_bot',
    'D_2017_vel_top',
    'D_2017_vel_mid',
    'D_2017_vel_bot',
    'D_2017_ctd_top',
    'D_2017_ctd_mid1',
    'D_2017_temp_mid2',
    'D_2017_ctd_bot',
    'D_2019_vel_top',
    'D_2019_vel_mid',
    'D_2019_vel_bot',
    'D_2019_ctd_top',
    'D_2019_ctd_mid1',
    'D_2019_temp_mid2',
    'D_2019_ctd_bot',
    'D_2020_vel_top',
    'D_2020_vel_mid',
    'D_2020_ctd_top',
    'D_2020_temp_mid1',
    'D_2020_ctd_mid2',
    'D_2021_vel_top',
    'D_2021_vel_mid',
    'D_2021_vel_bot',
    'D_2021_ctd_top',
    'D_2021_ctd_mid1',
    'D_2021_temp_mid2',
    'D_2021_ctd_bot',
    'D_2022_vel_top',
    'D_2022_vel_mid',
    'D_2022_ctd_top',
    'D_2022_ctd_mid1',
    'D_2022_temp_mid2',
    'D_2023_vel_mid',
    'D_2023_vel_bot',
    'D_2023_ctd_mid1',
    'D_2023_temp_mid2',
    'D_2023_ctd_bot',
    'D_2024_vel_mid',
    'D_2024_vel_bot',
    'D_2024_ctd_mid1',
    'D_2024_temp_mid2',
    'D_2024_ctd_bot'
]

In [ ]:
# investigate renaming variables 

D_2014_ctd_bot

### Using previous datatree 

In [ ]:
# load datatree 
dt = xr.open_datatree('TNB_mooring_datatree_12112025.nc')

In [ ]:
df = summarize_all(dt)

In [ ]:
df.query("mooring == 'LDEO'")

### Preliminary Analysis: Make Plots of Each Variable

#### Data tables to get quick look at data 

In [ ]:
df.query("mooring == 'D' & position == 'top' & instrument == 'ctd'")

In [ ]:
df.query("mooring == 'D' & position == 'mid2'")

In [ ]:
df.query("mooring == 'D' & position == 'mid1'")

In [ ]:
df.query("mooring == 'D' & position == 'bot'& instrument == 'ctd'")

In [ ]:
df.query("mooring == 'D' & instrument == 'vel'")

#### Salinity 

In [ ]:
years = df.query("mooring == 'D' & instrument == 'ctd' & position == 'top'")['year'].unique()

In [ ]:
years

In [ ]:
df.query("mooring == 'D' & instrument == 'ctd' & position == 'top'")['depth'].unique()

In [ ]:
years = df.query("mooring == 'D' & instrument == 'ctd' & position == 'top'")['year'].unique()
fig = plt.figure(figsize=(12,4))
plot_timeseries(years=years,var='S_0',mooring='D',instrument='ctd',position='top',color='tab:blue',label='S_0')
plot_timeseries(years=years,var='S_1',mooring='D',instrument='ctd',position='top',color='tab:green',label='S_1')
plot_timeseries(years=years,var='S_2',mooring='D',instrument='ctd',position='top',color='tab:red',label='S_2')

# get handles and labels for legend 
handles, labels = plt.gca().get_legend_handles_labels()
# only picks unique labels / handles 
unique = dict(zip(labels, handles))
# makes legend with unique keys only 
plt.legend(unique.values(), unique.keys())

# set limits on plot 
plt.ylim(34.2,34.95)
plt.ylabel('Salinity')
plt.suptitle('Salinity at Mooring D Top Sensor')

In [ ]:
df.query("mooring == 'D' & instrument == 'ctd' & position == 'mid1'")['year'].unique()

In [ ]:
df.query("mooring == 'D' & instrument == 'ctd' & position == 'mid1'")['depth'].unique()

In [ ]:
df.query("mooring == 'D' & instrument == 'ctd' & position == 'mid2'")['year'].unique()

In [ ]:
df.query("mooring == 'D' & instrument == 'ctd' & position == 'mid2'")['depth'].unique()

In [ ]:
years = df.query("mooring == 'D' & instrument == 'ctd' & position == 'mid1'")['year'].unique()
fig = plt.figure(figsize=(12,4))
plot_timeseries(years=years,var='S_0',mooring='D',instrument='ctd',position='mid1',color='tab:blue',label='S_0')
plot_timeseries(years=years,var='S_1',mooring='D',instrument='ctd',position='mid1',color='tab:green',label='S_1')
plot_timeseries(years=years,var='S_2',mooring='D',instrument='ctd',position='mid1',color='tab:red',label='S_2')

# year 2020 has ctd at mid 2 at ~922 meters, plotting here to complete the time series 
years = df.query("mooring == 'D' & instrument == 'ctd' & position == 'mid2'")['year'].unique()
plot_timeseries(years=years,var='S_0',mooring='D',instrument='ctd',position='mid2',color='tab:blue',label='S_0')
plot_timeseries(years=years,var='S_1',mooring='D',instrument='ctd',position='mid2',color='tab:green',label='S_1')
plot_timeseries(years=years,var='S_2',mooring='D',instrument='ctd',position='mid2',color='tab:red',label='S_2')

# get handles and labels for legend 
handles, labels = plt.gca().get_legend_handles_labels()
# only picks unique labels / handles 
unique = dict(zip(labels, handles))
# makes legend with unique keys only 
plt.legend(unique.values(), unique.keys())

# set limits on plot 
plt.ylim(34.6,34.95)
plt.ylabel('Salinity')
plt.suptitle('Salinity at Mooring D Mid1 Sensor')

In [ ]:
df.query("mooring == 'D' & instrument == 'ctd' & position == 'bot'")['year'].unique()

In [ ]:
df.query("mooring == 'D' & instrument == 'ctd' & position == 'bot'")['depth'].unique()

In [ ]:
years = df.query("mooring == 'D' & instrument == 'ctd' & position == 'bot'")['year'].unique()
fig = plt.figure(figsize=(12,4))
plot_timeseries(years=years,var='S_0',mooring='D',instrument='ctd',position='bot',color='tab:blue',label='S_0')
plot_timeseries(years=years,var='S_1',mooring='D',instrument='ctd',position='bot',color='tab:green',label='S_1')
plot_timeseries(years=years,var='S_2',mooring='D',instrument='ctd',position='bot',color='tab:red',label='S_2')

# get handles and labels for legend 
handles, labels = plt.gca().get_legend_handles_labels()
# only picks unique labels / handles 
unique = dict(zip(labels, handles))
# makes legend with unique keys only 
plt.legend(unique.values(), unique.keys())

# set limits on plot 
plt.ylim(34.6,34.95)
plt.ylabel('Salinity')
plt.suptitle('Salinity at Mooring D Bot Sensor')

##### Comments 

Level 0 and 2 are available. I should ues level 2 when availabe as this is corrected for CTD cast at mooring depoloyment and recovery. Level 2 is not despiked and there are definitely some suspect spikes that should likely be removed. Need to despike level 2 data and rename to sal (this will be final product that I use for analysis). Note that there is a lot of missing data at bottom sensor 


#### Temperature

In [ ]:
df.query("mooring == 'D' & instrument == 'ctd' & position == 'top'")['year'].unique()

In [ ]:
df.query("mooring == 'D' & instrument == 'ctd' & position == 'top'")['depth'].unique()

In [ ]:
years = df.query("mooring == 'D' & instrument == 'ctd' & position == 'top'")['year'].unique()
fig = plt.figure(figsize=(12,4))
plot_timeseries(years=years,var='T_0',mooring='D',instrument='ctd',position='top',color='tab:blue',label='T_0')
plot_timeseries(years=years,var='Theta_0',mooring='D',instrument='ctd',position='top',color='tab:orange',label='Theta_0')
plot_timeseries(years=years,var='T_1',mooring='D',instrument='ctd',position='top',color='tab:green',label='T_1')
plot_timeseries(years=years,var='T_2',mooring='D',instrument='ctd',position='top',color='tab:red',label='T_2')

# get handles and labels for legend 
handles, labels = plt.gca().get_legend_handles_labels()
# only picks unique labels / handles 
unique = dict(zip(labels, handles))
# makes legend with unique keys only 
plt.legend(unique.values(), unique.keys())

# set limits on plot 
plt.ylim(-2.2,-1.8)
plt.ylabel('Temp')
plt.suptitle('Temp at Mooring D Top Sensor')

In [ ]:
years = df.query("mooring == 'D' & instrument == 'ctd' & position == 'mid1'")['year'].unique()
fig = plt.figure(figsize=(12,4))
plot_timeseries(years=years,var='T_0',mooring='D',instrument='ctd',position='mid1',color='tab:blue',label='T_0')
plot_timeseries(years=years,var='Theta_0',mooring='D',instrument='ctd',position='mid1',color='tab:orange',label='Theta_0')
plot_timeseries(years=years,var='T_1',mooring='D',instrument='ctd',position='mid1',color='tab:green',label='T_1')
plot_timeseries(years=years,var='T_2',mooring='D',instrument='ctd',position='mid1',color='tab:red',label='T_2')

# account for temp collected by thermistors, not ctds 
years = df.query("mooring == 'D' & instrument == 'temp' & position == 'mid1'")['year'].unique()
plot_timeseries(years=years,var='T_0',mooring='D',instrument='temp',position='mid1',color='tab:blue',label='T_0')
plot_timeseries(years=years,var='Theta_0',mooring='D',instrument='temp',position='mid1',color='tab:orange',label='Theta_0')
plot_timeseries(years=years,var='T_1',mooring='D',instrument='temp',position='mid1',color='tab:green',label='T_1')
plot_timeseries(years=years,var='T_2',mooring='D',instrument='temp',position='mid1',color='tab:red',label='T_2')

# get handles and labels for legend 
handles, labels = plt.gca().get_legend_handles_labels()
# only picks unique labels / handles 
unique = dict(zip(labels, handles))
# makes legend with unique keys only 
plt.legend(unique.values(), unique.keys())

# set limits on plot 
plt.ylim(-2.5,-1.85)
plt.ylabel('Temp')
plt.suptitle('Temp at Mooring D mid1 Sensor')

In [ ]:
years = df.query("mooring == 'D' & instrument == 'ctd' & position == 'mid2'")['year'].unique()
fig = plt.figure(figsize=(12,4))
plot_timeseries(years=years,var='T_0',mooring='D',instrument='ctd',position='mid2',color='tab:blue',label='T_0')
plot_timeseries(years=years,var='Theta_0',mooring='D',instrument='ctd',position='mid2',color='tab:orange',label='Theta_0')
plot_timeseries(years=years,var='T_1',mooring='D',instrument='ctd',position='mid2',color='tab:green',label='T_1')
plot_timeseries(years=years,var='T_2',mooring='D',instrument='ctd',position='mid2',color='tab:red',label='T_2')

# account for temp collected by thermistors, not ctds 
years = df.query("mooring == 'D' & instrument == 'temp' & position == 'mid2'")['year'].unique()
plot_timeseries(years=years,var='T_0',mooring='D',instrument='temp',position='mid2',color='tab:blue',label='T_0')
plot_timeseries(years=years,var='Theta_0',mooring='D',instrument='temp',position='mid2',color='tab:orange',label='Theta_0')
plot_timeseries(years=years,var='T_1',mooring='D',instrument='temp',position='mid2',color='tab:green',label='T_1')
plot_timeseries(years=years,var='T_2',mooring='D',instrument='temp',position='mid2',color='tab:red',label='T_2')

# get handles and labels for legend 
handles, labels = plt.gca().get_legend_handles_labels()
# only picks unique labels / handles 
unique = dict(zip(labels, handles))
# makes legend with unique keys only 
plt.legend(unique.values(), unique.keys())

# set limits on plot 
plt.ylim(-2,-1.85)
plt.ylabel('Temp')
plt.suptitle('Temp at Mooring D mid2 Sensor')

In [ ]:
years = df.query("mooring == 'D' & instrument == 'ctd' & position == 'bot'")['year'].unique()
fig = plt.figure(figsize=(12,4))
plot_timeseries(years=years,var='T_0',mooring='D',instrument='ctd',position='bot',color='tab:blue',label='T_0')
plot_timeseries(years=years,var='Theta_0',mooring='D',instrument='ctd',position='bot',color='tab:orange',label='Theta_0')
plot_timeseries(years=years,var='T_1',mooring='D',instrument='ctd',position='bot',color='tab:green',label='T_1')
plot_timeseries(years=years,var='T_2',mooring='D',instrument='ctd',position='bot',color='tab:red',label='T_2')

# get handles and labels for legend 
handles, labels = plt.gca().get_legend_handles_labels()
# only picks unique labels / handles 
unique = dict(zip(labels, handles))
# makes legend with unique keys only 
plt.legend(unique.values(), unique.keys())

# set limits on plot 
plt.ylim(-2,-1.85)
plt.ylabel('Temp')
plt.suptitle('Temp at Mooring D bot Sensor')

#### Velocity: zonal, meridional, speed 

In [ ]:
fig = plt.figure(figsize=(12,4))
years = df.query("mooring == 'D' & instrument == 'vel' & position == 'top'")['year'].unique()
plot_timeseries(years=years,var='U_0',mooring='D',instrument='vel',position='top',color='tab:blue',label='U_0 top')
# get handles and labels for legend 
handles, labels = plt.gca().get_legend_handles_labels()
# only picks unique labels / handles 
unique = dict(zip(labels, handles))
# makes legend with unique keys only 
plt.legend(unique.values(), unique.keys())
# set limits on plot 
plt.ylim(-50,50)
plt.ylabel('Zonal Velocity (m/s)')
plt.suptitle('Zonal Velocity at Mooring D Top Sensor')

In [ ]:
fig = plt.figure(figsize=(12,4))
years = df.query("mooring == 'D' & instrument == 'vel' & position == 'mid'")['year'].unique()
plot_timeseries(years=years,var='U_0',mooring='D',instrument='vel',position='mid',color='tab:blue',label='U_0 mid')
# get handles and labels for legend 
handles, labels = plt.gca().get_legend_handles_labels()
# only picks unique labels / handles 
unique = dict(zip(labels, handles))
# makes legend with unique keys only 
plt.legend(unique.values(), unique.keys())
# set limits on plot 
plt.ylim(-50,50)
plt.ylabel('Zonal Velocity (m/s)')
plt.suptitle('Zonal Velocity at Mooring D mid Sensor')

In [ ]:
fig = plt.figure(figsize=(12,4))
years = df.query("mooring == 'D' & instrument == 'vel' & position == 'bot'")['year'].unique()
plot_timeseries(years=years,var='U_0',mooring='D',instrument='vel',position='bot',color='tab:blue',label='U_0 bot')
# get handles and labels for legend 
handles, labels = plt.gca().get_legend_handles_labels()
# only picks unique labels / handles 
unique = dict(zip(labels, handles))
# makes legend with unique keys only 
plt.legend(unique.values(), unique.keys())
# set limits on plot 
plt.ylim(-50,50)
plt.ylabel('Zonal Velocity (m/s)')
plt.suptitle('Zonal Velocity at Mooring D bot Sensor')

In [ ]:
fig = plt.figure(figsize=(12,4))
years = df.query("mooring == 'D' & instrument == 'vel' & position == 'top'")['year'].unique()
plot_timeseries(years=years,var='U_0',mooring='D',instrument='vel',position='top',color='tab:blue',label='U_0 top')

years = df.query("mooring == 'D' & instrument == 'vel' & position == 'mid'")['year'].unique()
plot_timeseries(years=years,var='U_0',mooring='D',instrument='vel',position='mid',color='tab:green',label='U_0 mid')

years = df.query("mooring == 'D' & instrument == 'vel' & position == 'bot'")['year'].unique()
plot_timeseries(years=years,var='U_0',mooring='D',instrument='vel',position='bot',color='tab:orange',label='U_0 bot')
# get handles and labels for legend 
handles, labels = plt.gca().get_legend_handles_labels()
# only picks unique labels / handles 
unique = dict(zip(labels, handles))
# makes legend with unique keys only 
plt.legend(unique.values(), unique.keys())
# set limits on plot 
plt.ylim(-50,50)
plt.ylabel('Zonal Velocity (m/s)')
plt.suptitle('Zonal Velocity at Mooring D')

In [ ]:
fig = plt.figure(figsize=(12,4))
years = df.query("mooring == 'D' & instrument == 'vel' & position == 'top'")['year'].unique()
plot_timeseries(years=years,var='V_0',mooring='D',instrument='vel',position='top',color='tab:blue',label='V_0 top')
# get handles and labels for legend 
handles, labels = plt.gca().get_legend_handles_labels()
# only picks unique labels / handles 
unique = dict(zip(labels, handles))
# makes legend with unique keys only 
plt.legend(unique.values(), unique.keys())
# set limits on plot 
plt.ylim(-50,50)
plt.ylabel('Meridional Velocity (m/s)')
plt.suptitle('Meridional Velocity at Mooring D Top Sensor')

In [ ]:
fig = plt.figure(figsize=(12,4))
years = df.query("mooring == 'D' & instrument == 'vel' & position == 'mid'")['year'].unique()
plot_timeseries(years=years,var='V_0',mooring='D',instrument='vel',position='mid',color='tab:blue',label='V_0 mid')
# get handles and labels for legend 
handles, labels = plt.gca().get_legend_handles_labels()
# only picks unique labels / handles 
unique = dict(zip(labels, handles))
# makes legend with unique keys only 
plt.legend(unique.values(), unique.keys())
# set limits on plot 
plt.ylim(-50,50)
plt.ylabel('Meridional Velocity (m/s)')
plt.suptitle('Meridional Velocity at Mooring D mid Sensor')

In [ ]:
fig = plt.figure(figsize=(12,4))
years = df.query("mooring == 'D' & instrument == 'vel' & position == 'bot'")['year'].unique()
plot_timeseries(years=years,var='V_0',mooring='D',instrument='vel',position='bot',color='tab:blue',label='V_0 bot')
# get handles and labels for legend 
handles, labels = plt.gca().get_legend_handles_labels()
# only picks unique labels / handles 
unique = dict(zip(labels, handles))
# makes legend with unique keys only 
plt.legend(unique.values(), unique.keys())
# set limits on plot 
plt.ylim(-50,50)
plt.ylabel('Meridional Velocity (m/s)')
plt.suptitle('Meridional Velocity at Mooring D bot Sensor')

In [ ]:
fig = plt.figure(figsize=(12,4))
years = df.query("mooring == 'D' & instrument == 'vel' & position == 'top'")['year'].unique()
plot_timeseries(years=years,var='V_0',mooring='D',instrument='vel',position='top',color='tab:blue',label='V_0 top')

years = df.query("mooring == 'D' & instrument == 'vel' & position == 'mid'")['year'].unique()
plot_timeseries(years=years,var='V_0',mooring='D',instrument='vel',position='mid',color='tab:green',label='V_0 mid')

years = df.query("mooring == 'D' & instrument == 'vel' & position == 'bot'")['year'].unique()
plot_timeseries(years=years,var='V_0',mooring='D',instrument='vel',position='bot',color='tab:orange',label='V_0 bot')
# get handles and labels for legend 
handles, labels = plt.gca().get_legend_handles_labels()
# only picks unique labels / handles 
unique = dict(zip(labels, handles))
# makes legend with unique keys only 
plt.legend(unique.values(), unique.keys())
# set limits on plot 
plt.ylim(-50,50)
plt.ylabel('Meridional Velocity (m/s)')
plt.suptitle('Meridional Velocity at Mooring D')

In [ ]:
fig = plt.figure(figsize=(12,4))
years = df.query("mooring == 'D' & instrument == 'vel' & position == 'top'")['year'].unique()
plot_timeseries(years=years,var='Mod_0',mooring='D',instrument='vel',position='top',color='tab:blue',label='Mod_0 top')
# get handles and labels for legend 
handles, labels = plt.gca().get_legend_handles_labels()
# only picks unique labels / handles 
unique = dict(zip(labels, handles))
# makes legend with unique keys only 
plt.legend(unique.values(), unique.keys())
# set limits on plot 
plt.ylim(0,50)
plt.ylabel('Speed (m/s)')
plt.suptitle('Speed at Mooring D Top Sensor')

In [ ]:
fig = plt.figure(figsize=(12,4))
years = df.query("mooring == 'D' & instrument == 'vel' & position == 'mid'")['year'].unique()
plot_timeseries(years=years,var='Mod_0',mooring='D',instrument='vel',position='mid',color='tab:blue',label='Mod_0 mid')
# get handles and labels for legend 
handles, labels = plt.gca().get_legend_handles_labels()
# only picks unique labels / handles 
unique = dict(zip(labels, handles))
# makes legend with unique keys only 
plt.legend(unique.values(), unique.keys())
# set limits on plot 
plt.ylim(0,50)
plt.ylabel('Speed (m/s)')
plt.suptitle('Speed at Mooring D mid Sensor')

In [ ]:
fig = plt.figure(figsize=(12,4))
years = df.query("mooring == 'D' & instrument == 'vel' & position == 'bot'")['year'].unique()
plot_timeseries(years=years,var='Mod_0',mooring='D',instrument='vel',position='bot',color='tab:blue',label='Mod_0 bot')
# get handles and labels for legend 
handles, labels = plt.gca().get_legend_handles_labels()
# only picks unique labels / handles 
unique = dict(zip(labels, handles))
# makes legend with unique keys only 
plt.legend(unique.values(), unique.keys())
# set limits on plot 
plt.ylim(0,50)
plt.ylabel('Speed (m/s)')
plt.suptitle('Speed at Mooring D bot Sensor')

In [ ]:
fig = plt.figure(figsize=(12,4))
years = df.query("mooring == 'D' & instrument == 'vel' & position == 'top'")['year'].unique()
plot_timeseries(years=years,var='Mod_0',mooring='D',instrument='vel',position='top',color='tab:blue',label='Mod_0 top')

years = df.query("mooring == 'D' & instrument == 'vel' & position == 'mid'")['year'].unique()
plot_timeseries(years=years,var='Mod_0',mooring='D',instrument='vel',position='mid',color='tab:green',label='Mod_0 mid')

years = df.query("mooring == 'D' & instrument == 'vel' & position == 'bot'")['year'].unique()
plot_timeseries(years=years,var='Mod_0',mooring='D',instrument='vel',position='bot',color='tab:orange',label='Mod_0 bot')
# get handles and labels for legend 
handles, labels = plt.gca().get_legend_handles_labels()
# only picks unique labels / handles 
unique = dict(zip(labels, handles))
# makes legend with unique keys only 
plt.legend(unique.values(), unique.keys())
# set limits on plot 
plt.ylim(0,50)
plt.ylabel('Speed (m/s)')
plt.suptitle('Speed at Mooring D')

### Despike Data

Going through each salinity time series to despike. Will despike using a modified z-score and will pick the threshold after visual inspection. 

In [ ]:
# make function for modified z-score 
def modified_zscore(data):
    """
    """

    # calculate median 
    median = np.nanmedian(data)
    MAD = median_abs_deviation(data, nan_policy='omit')
    modified_zscore = 0.6745* (data-median)/MAD

    return modified_zscore

def despike_by_zscore(data, zscore, threshold):
    """  
    """

    indx = np.where(np.abs(zscore) > threshold)
    despiked = data.copy()
    despiked[indx] = np.nan
    return despiked

In [ ]:
def get_outliers(data, threshold):
    """
    Returns an xarray Dataset containing:
      - outlier_mask (True where flagged)
      - zscore
      - outlier_values
    """
    z = modified_zscore(data)
    mask = np.abs(z) > threshold

    return xr.Dataset({
        "outlier_mask": mask,
        "zscore": z.where(mask),
        "outlier_values": data.where(mask)
    })


In [ ]:
def seasonal_despike(da, threshold=3, season_def=None):
    """
    Performs seasonal modified z-score despiking on an xarray DataArray.

    Returns:
        despiked : DataArray (same shape as input, outliers replaced with NaN)
        flagged_times : list of np.datetime64 timestamps flagged as outliers
    """

    if season_def is None:
        season_def = {
            "DJF": [12,1,2],
            "MAM": [3,4,5],
            "JJA": [6,7,8],
            "SON": [9,10,11],
        }

    despiked = da.copy()
    flagged_times = []

    for season, months in season_def.items():

        # subset of data in this season (drop=False keeps full alignment)
        subset = da.where(da.time.dt.month.isin(months), drop=False)

        # compute modified z-score
        median = np.nanmedian(subset)
        MAD = median_abs_deviation(subset, nan_policy='omit')

        # if MAD = 0 → no variability → skip season safely
        if MAD == 0 or np.isnan(MAD):
            continue

        z = 0.6745 * (subset - median) / MAD

        # outlier mask
        mask = np.abs(z) > threshold

        # store timestamps of outliers
        flagged = subset.time.where(mask).dropna("time")
        flagged_times.extend(flagged.values)

        # apply despiking
        despiked = despiked.where(~mask)

    return despiked, flagged_times


In [ ]:
def get_despiked_data_from_datatree(year,instrument,position,var):
    """ 
    """

    data = dt['D'][year][instrument][position][var]
    mod_zscore = modified_zscore(data)
    data_despiked_2 = despike_by_zscore(data,mod_zscore,2)
    data_despiked_3 = despike_by_zscore(data,mod_zscore,3)
    data_despiked_4 = despike_by_zscore(data,mod_zscore,4)
    data_despiked_5 = despike_by_zscore(data,mod_zscore,5)
    data_despiked_seasonal, flagged_times_seasonal = seasonal_despike(data,threshold=4,season_def=None)

    return data, data_despiked_2, data_despiked_3, data_despiked_4, data_despiked_5, data_despiked_seasonal, flagged_times_seasonal

In [ ]:
def get_despiked_data(data):
    """ 
    """

    # data = dt['D'][year][instrument][position][var]
    mod_zscore = modified_zscore(data)
    data_despiked_2 = despike_by_zscore(data,mod_zscore,2)
    data_despiked_3 = despike_by_zscore(data,mod_zscore,3)
    data_despiked_4 = despike_by_zscore(data,mod_zscore,4)
    data_despiked_5 = despike_by_zscore(data,mod_zscore,5)
    data_despiked_seasonal, flagged_times_seasonal = seasonal_despike(data,threshold=4,season_def=None)

    return data, data_despiked_2, data_despiked_3, data_despiked_4, data_despiked_5, data_despiked_seasonal, flagged_times_seasonal

In [ ]:
def plot_despiked_data():
    fig = plt.figure(figsize=(12,20),constrained_layout=True)

    ax0 = fig.add_subplot(5,1,1)
    plt.plot(data.time,data,label='Originial Data')
    plt.plot(data_despiked_2.time,data_despiked_2,label='Despiked, threshold=2',color='tab:green')
    plt.legend()

    ax1 = fig.add_subplot(5,1,2)
    plt.plot(data.time,data,label='Originial Data')
    plt.plot(data_despiked_3.time,data_despiked_3,label='Despiked, threshold=3',color='tab:purple')
    plt.legend()

    ax2 = fig.add_subplot(5,1,3)
    plt.plot(data.time,data,label='Originial Data')
    plt.plot(data_despiked_4.time,data_despiked_4,label='Despiked, threshold=4',color='tab:orange')
    plt.legend()

    ax3 = fig.add_subplot(5,1,4)
    plt.plot(data.time,data,label='Originial Data')
    plt.plot(data_despiked_5.time,data_despiked_5,label='Despiked, threshold=5',color='tab:red')
    plt.legend()

    ax4 = fig.add_subplot(5,1,5)
    plt.plot(data.time,data,label='Originial Data')
    plt.plot(data_despiked_seasonal.time,data_despiked_seasonal,label='Despiked Seasonally, threshold=4',color='tab:brown')
    plt.legend()

    return ax0,ax1,ax2,ax3,ax4

In [ ]:
df.query("mooring == 'D' & position == 'top' & instrument == 'ctd'")

#### Despike Mooring D Salinity Top Sensor 

##### 2014

In [ ]:
# get despiked data at various zscore thresholds 
data, data_despiked_2, data_despiked_3, data_despiked_4, data_despiked_5, data_despiked_seasonal, flagged_times_seasonal = get_despiked_data(data=D_2014_ctd_top.S_2)

In [ ]:
# get time stamps of outliers 
outliers_2 = get_outliers(data=data,threshold=2)
outliers_3 = get_outliers(data=data,threshold=3)
outliers_4 = get_outliers(data=data,threshold=4)
outliers_5 = get_outliers(data=data,threshold=5)

flagged_times_2 = outliers_2.outlier_values.dropna(dim="time").time.values
flagged_times_3 = outliers_3.outlier_values.dropna(dim="time").time.values
flagged_times_4 = outliers_4.outlier_values.dropna(dim="time").time.values
flagged_times_5 = outliers_5.outlier_values.dropna(dim="time").time.values

In [ ]:
flagged_times_seasonal

In [ ]:
# plot the various despike data 
plot_despiked_data()
plt.suptitle('Mooring D Top S_2 2014')

In [ ]:
data_despiked_seasonal

In [ ]:
D_2014_ctd_top['sal'] = ('time',data_despiked_seasonal.data)

In [ ]:
# will use the seaonsal despike for sal data 
# dt['D']['2014']['ctd']['top']['sal'] = data_despiked_seasonal

In [ ]:
# dt['D']['2014']['ctd']['top']

##### 2016

In [ ]:
data, data_despiked_2, data_despiked_3, data_despiked_4, data_despiked_5, data_despiked_seasonal, flagged_times_seasonal = get_despiked_data(data = D_2016_ctd_top.S_0)

In [ ]:
ax0, ax1, ax2, ax3, ax4 = plot_despiked_data()
ax0.set_ylim(34.4,34.6)
ax1.set_ylim(34.4,34.6)
ax2.set_ylim(34.4,34.6)
ax3.set_ylim(34.4,34.6)
ax4.set_ylim(33.8,34.6)
plt.suptitle('Mooring D Top S_0 2016')

No S_2 data for 2016 so skipping this one - in orginal plot it is clear that S_0 data is lower than S_2 and likely needs to be corrected somehow 

##### 2017

In [ ]:
data, data_despiked_2, data_despiked_3, data_despiked_4, data_despiked_5, data_despiked_seasonal, flagged_times_seasonal = get_despiked_data(data = D_2017_ctd_top.S_2)

In [ ]:
ax0, ax1, ax2, ax3, ax4 = plot_despiked_data()
# ax0.set_ylim(34.4,34.6)
# ax1.set_ylim(34.4,34.6)
# ax2.set_ylim(34.4,34.6)
# ax3.set_ylim(34.4,34.6)
# ax4.set_ylim(33.8,34.6)
plt.suptitle('Mooring D Top S_2 2017')

In [ ]:
flagged_times_seasonal

In [ ]:
D_2017_ctd_top['sal'] = ('time',data_despiked_seasonal.data)

In [ ]:
# will despike using seasonal despiking method 
# dt['D']['2017']['ctd']['top']['sal'] = data_despiked_seasonal

##### 2019

In [ ]:
data, data_despiked_2, data_despiked_3, data_despiked_4, data_despiked_5, data_despiked_seasonal, flagged_times_seasonal = get_despiked_data(data = D_2019_ctd_top.S_2)

In [ ]:
ax0, ax1, ax2, ax3, ax4 = plot_despiked_data()
# ax0.set_ylim(34.4,34.6)
# ax1.set_ylim(34.4,34.6)
# ax2.set_ylim(34.4,34.6)
# ax3.set_ylim(34.4,34.6)
# ax4.set_ylim(33.8,34.6)
plt.suptitle('Mooring D Top S_2 2019')

In [ ]:
D_2019_ctd_top['sal'] = ('time',data_despiked_3.data)

In [ ]:
# despike using threshold of 3 
# dt['D']['2019']['ctd']['top']['sal'] = data_despiked_3

##### 2020

In [ ]:
data, data_despiked_2, data_despiked_3, data_despiked_4, data_despiked_5, data_despiked_seasonal, flagged_times_seasonal = get_despiked_data(data = D_2020_ctd_top.S_2)

In [ ]:
ax0, ax1, ax2, ax3, ax4 = plot_despiked_data()
# ax0.set_ylim(34.4,34.6)
# ax1.set_ylim(34.4,34.6)
# ax2.set_ylim(34.4,34.6)
# ax3.set_ylim(34.4,34.6)
# ax4.set_ylim(33.8,34.6)
plt.suptitle('Mooring D Top S_2 2020')

In [ ]:
sal = D_2020_ctd_top.sel(time=slice('2020-02-01T16','2021-01-07T20')).S_2

In [ ]:
D_2020_ctd_top = D_2020_ctd_top.sel(time=slice('2020-02-01T16','2021-01-07T20'))
D_2020_ctd_top['sal'] = ('time',sal.data) 

In [ ]:
# really just need to cut of ends of 2020 data, the rest of the data looks like it could be real 
# ds = dt['D']['2020']['ctd']['top'].sel(time=slice('2020-02-01T16','2021-01-07T20'))
# dt['D']['2020']['ctd']['top']['sal'] = ds['S_2'] 

##### 2021

In [ ]:
data, data_despiked_2, data_despiked_3, data_despiked_4, data_despiked_5, data_despiked_seasonal, flagged_times_seasonal = get_despiked_data(data = D_2021_ctd_top.S_2)

In [ ]:
ax0, ax1, ax2, ax3, ax4 = plot_despiked_data()
# ax0.set_ylim(34.4,34.6)
# ax1.set_ylim(34.4,34.6)
# ax2.set_ylim(34.4,34.6)
# ax3.set_ylim(34.4,34.6)
# ax4.set_ylim(33.8,34.6)
plt.suptitle('Mooring D Top S_2 2021')

In [ ]:
D_2021_ctd_top['sal'] = ('time',D_2021_ctd_top.S_2.data)

In [ ]:
# data looks fine and does not need to be despiked 
# dt['D']['2021']['ctd']['top']['sal'] = dt['D']['2021']['ctd']['top']['S_2'] 

##### 2022

In [ ]:
data, data_despiked_2, data_despiked_3, data_despiked_4, data_despiked_5, data_despiked_seasonal, flagged_times_seasonal = get_despiked_data(data = D_2022_ctd_top.S_2)

In [ ]:
ax0, ax1, ax2, ax3, ax4 = plot_despiked_data()
# ax0.set_ylim(34.4,34.6)
# ax1.set_ylim(34.4,34.6)
# ax2.set_ylim(34.4,34.6)
# ax3.set_ylim(34.4,34.6)
# ax4.set_ylim(33.8,34.6)
plt.suptitle('Mooring D Top S_2 2022')

In [ ]:
# get spikes around 2022-06 and 2022-11
time0 = np.datetime64('2022-11-10T03:40:00.000000000')
time1 = np.datetime64('2022-06-08T06:40:00.000000000')


In [ ]:
D_2022_ctd_top['sal'] = ('time',D_2022_ctd_top.S_2.data)

In [ ]:
D_2022_ctd_top['sal'].loc[dict(time=time0)] = np.nan
D_2022_ctd_top['sal'].loc[dict(time=time1)] = np.nan

In [ ]:
fig = plt.figure()
D_2022_ctd_top.sal.plot()

In [ ]:
# leaving salinity as it for now, might want to remove spike around 2022-06 and 2022-22 later 
# dt['D']['2022']['ctd']['top']['sal'] = dt['D']['2022']['ctd']['top']['S_2'].copy()

In [ ]:
# da = dt['D']['2022']['ctd']['top']['sal']
# da.loc[dict(time=time0)] = np.nan
# da.loc[dict(time=time1)] = np.nan

In [ ]:
# make plot of S2 and sal (i.e before and after despiking)
# years = df.query("mooring == 'D' & instrument == 'ctd' & position == 'top'")['year'].unique()
# fig = plt.figure(figsize=(12,4))
# # plot_timeseries(years=years,var='S_0',mooring='D',instrument='ctd',position='top',color='tab:blue',label='S_0')
# # plot_timeseries(years=years,var='S_1',mooring='D',instrument='ctd',position='top',color='tab:green',label='S_1')
# plot_timeseries(years=years,var='S_2',mooring='D',instrument='ctd',position='top',color='tab:red',label='S_2')
# # plot_timeseries(years=years,var='sal',mooring='D',instrument='ctd',position='top',color='tab:blue',label='sal')

# # get handles and labels for legend 
# handles, labels = plt.gca().get_legend_handles_labels()
# # only picks unique labels / handles 
# unique = dict(zip(labels, handles))
# # makes legend with unique keys only 
# plt.legend(unique.values(), unique.keys())

# # set limits on plot 
# plt.ylim(34.2,34.95)
# plt.ylabel('Salinity')
# plt.suptitle('Salinity at Mooring D Top Sensor')

#### Despike Mooring D Salinity Mid1 Sensor 

In [ ]:
df.query("mooring == 'D' & position == 'mid1'")

##### 2017

In [ ]:
data, data_despiked_2, data_despiked_3, data_despiked_4, data_despiked_5, data_despiked_seasonal, flagged_times_seasonal = get_despiked_data(data = D_2017_ctd_mid1.S_2)

In [ ]:
ax0, ax1, ax2, ax3, ax4 = plot_despiked_data()
# ax0.set_ylim(34.4,34.6)
# ax1.set_ylim(34.4,34.6)
# ax2.set_ylim(34.4,34.6)
# ax3.set_ylim(34.4,34.6)
# ax4.set_ylim(33.8,34.6)
plt.suptitle('Mooring D Mid1 S_2 2017')

In [ ]:
D_2017_ctd_mid1['sal'] = ('time',D_2017_ctd_mid1.S_2.data)

In [ ]:
# nothing to despike 
# dt['D']['2017']['ctd']['mid1']['sal'] = dt['D']['2017']['ctd']['mid1']['S_2'].copy()

##### 2019

In [ ]:
data, data_despiked_2, data_despiked_3, data_despiked_4, data_despiked_5, data_despiked_seasonal, flagged_times_seasonal = get_despiked_data(data = D_2019_ctd_mid1.S_2)

In [ ]:
ax0, ax1, ax2, ax3, ax4 = plot_despiked_data()
# ax0.set_ylim(34.4,34.6)
# ax1.set_ylim(34.4,34.6)
# ax2.set_ylim(34.4,34.6)
# ax3.set_ylim(34.4,34.6)
# ax4.set_ylim(33.8,34.6)
plt.suptitle('Mooring D Mid1 S_2 2019')

In [ ]:
D_2019_ctd_mid1['sal'] = ('time', data_despiked_5.data)

In [ ]:
# despiking using modified z score with threshold of 5 
# dt['D']['2019']['ctd']['mid1']['sal'] = data_despiked_5

In [ ]:
D_2019_ctd_mid1

##### 2021

In [ ]:
data, data_despiked_2, data_despiked_3, data_despiked_4, data_despiked_5, data_despiked_seasonal, flagged_times_seasonal = get_despiked_data(data = D_2021_ctd_mid1.S_2)

In [ ]:
ax0, ax1, ax2, ax3, ax4 = plot_despiked_data()
# ax0.set_ylim(34.4,34.6)
# ax1.set_ylim(34.4,34.6)
# ax2.set_ylim(34.4,34.6)
# ax3.set_ylim(34.4,34.6)
# ax4.set_ylim(33.8,34.6)
plt.suptitle('Mooring D Mid1 S_2 2021')

In [ ]:
outliers_2 = get_outliers(data=data,threshold=2)
outliers_3 = get_outliers(data=data,threshold=3)
outliers_4 = get_outliers(data=data,threshold=4)
outliers_5 = get_outliers(data=data,threshold=5)

flagged_times_2 = outliers_2.outlier_values.dropna(dim="time").time.values
flagged_times_3 = outliers_3.outlier_values.dropna(dim="time").time.values
flagged_times_4 = outliers_4.outlier_values.dropna(dim="time").time.values
flagged_times_5 = outliers_5.outlier_values.dropna(dim="time").time.values

In [ ]:
flagged_times_5[0]

In [ ]:
time0 = np.datetime64('2022-01-24T22:10:00.000000000')
time1 = np.datetime64('2021-03-30T18:40:00.000000000')
time2 = flagged_times_5[0]


In [ ]:
D_2021_ctd_mid1['sal'] = ('time',D_2021_ctd_mid1.S_2.data)

In [ ]:
D_2021_ctd_mid1['sal'].loc[dict(time=time0)] = np.nan
D_2021_ctd_mid1['sal'].loc[dict(time=time1)] = np.nan
D_2021_ctd_mid1['sal'].loc[dict(time=time2)] = np.nan

In [ ]:
# dt['D']['2021']['ctd']['mid1']['sal'] = dt['D']['2021']['ctd']['mid1']['S_2'].copy()

In [ ]:
# da = dt['D']['2021']['ctd']['mid1']['sal']
# da.loc[dict(time=time0)] = np.nan
# da.loc[dict(time=time1)] = np.nan
# da.loc[dict(time=time2)] = np.nan

In [ ]:
fig = plt.figure()
D_2021_ctd_mid1.sal.plot()

In [ ]:
# fig = plt.figure()
# dt['D']['2021']['ctd']['mid1']['sal'].plot()

##### 2022

In [ ]:
data, data_despiked_2, data_despiked_3, data_despiked_4, data_despiked_5, data_despiked_seasonal, flagged_times_seasonal = get_despiked_data(data = D_2022_ctd_mid1.S_2)

In [ ]:
ax0, ax1, ax2, ax3, ax4 = plot_despiked_data()
# ax0.set_ylim(34.4,34.6)
# ax1.set_ylim(34.4,34.6)
# ax2.set_ylim(34.4,34.6)
# ax3.set_ylim(34.4,34.6)
# ax4.set_ylim(33.8,34.6)
plt.suptitle('Mooring D Mid1 S_2 2022')

In [ ]:
D_2022_ctd_mid1 = D_2022_ctd_mid1.isel(time=slice(0,-1))

In [ ]:
D_2022_ctd_mid1['sal'] = ('time',D_2022_ctd_mid1.S_2.data)

In [ ]:
# # just want to remove last point 
# dt['D']['2022']['ctd']['mid1'] = dt['D']['2022']['ctd']['mid1'].copy().isel(time=slice(0,-1))
# dt['D']['2022']['ctd']['mid1']['sal'] = dt['D']['2022']['ctd']['mid1']['S_2']

##### 2023

In [ ]:
data, data_despiked_2, data_despiked_3, data_despiked_4, data_despiked_5, data_despiked_seasonal, flagged_times_seasonal = get_despiked_data(data = D_2023_ctd_mid1.S_2)

In [ ]:
ax0, ax1, ax2, ax3, ax4 = plot_despiked_data()
# ax0.set_ylim(34.4,34.6)
# ax1.set_ylim(34.4,34.6)
# ax2.set_ylim(34.4,34.6)
# ax3.set_ylim(34.4,34.6)
# ax4.set_ylim(33.8,34.6)
plt.suptitle('Mooring D Mid1 S_2 2023')

In [ ]:
D_2023_ctd_mid1['sal'] = ('time', D_2023_ctd_mid1.S_2.data)

In [ ]:
# nothing to despike 
# dt['D']['2023']['ctd']['mid1']['sal'] = dt['D']['2023']['ctd']['mid1']['S_2'].copy()

##### 2024

In [ ]:
data, data_despiked_2, data_despiked_3, data_despiked_4, data_despiked_5, data_despiked_seasonal, flagged_times_seasonal = get_despiked_data(data = D_2024_ctd_mid1.S_2)

In [ ]:
ax0, ax1, ax2, ax3, ax4 = plot_despiked_data()
# ax0.set_ylim(34.4,34.6)
# ax1.set_ylim(34.4,34.6)
# ax2.set_ylim(34.4,34.6)
# ax3.set_ylim(34.4,34.6)
# ax4.set_ylim(33.8,34.6)
plt.suptitle('Mooring D Mid1 S_2 2024')

In [ ]:
D_2024_ctd_mid1['sal'] = ('time', D_2024_ctd_mid1.S_2.data)

In [ ]:
# nothing to despike 
# dt['D']['2024']['ctd']['mid1']['sal'] = dt['D']['2024']['ctd']['mid1']['S_2'].copy()

##### 2020

2020 had ctd sensor placed at mid2 location so adding it in to make time series continuous 

In [ ]:
data, data_despiked_2, data_despiked_3, data_despiked_4, data_despiked_5, data_despiked_seasonal, flagged_times_seasonal = get_despiked_data(data = D_2020_ctd_mid2.S_2)

In [ ]:
ax0, ax1, ax2, ax3, ax4 = plot_despiked_data()
# ax0.set_ylim(34.4,34.6)
# ax1.set_ylim(34.4,34.6)
# ax2.set_ylim(34.4,34.6)
# ax3.set_ylim(34.4,34.6)
# ax4.set_ylim(33.8,34.6)
plt.suptitle('Mooring D Mid2 S_2 2020')

In [ ]:
# to remove 
# time0 = np.datetime64('2020-02-01T13:29:00.000000000')
time1 = np.datetime64('2021-01-07T21:29:00.000000000')
time2 = np.datetime64('2020-06-10T21:29:00.000000000')


In [ ]:
D_2020_ctd_mid2['sal'] = ('time',D_2020_ctd_mid2.S_2.data)
D_2020_ctd_mid2 = D_2020_ctd_mid2.sel(time=slice('2020-02-01T14:45',None))

# D_2020_ctd_mid2['sal'].loc[dict(time=time0)] = np.nan
D_2020_ctd_mid2['sal'].loc[dict(time=time1)] = np.nan
D_2020_ctd_mid2['sal'].loc[dict(time=time2)] = np.nan

In [ ]:
# dt['D']['2020']['ctd']['mid2']['sal'] = dt['D']['2020']['ctd']['mid2']['S_2'].copy()

In [ ]:
# da = dt['D']['2020']['ctd']['mid2']['sal']
# da.loc[dict(time=time0)] = np.nan
# da.loc[dict(time=time1)] = np.nan
# da.loc[dict(time=time2)] = np.nan

In [ ]:
fig = plt.figure()
D_2020_ctd_mid2.sal.plot()

In [ ]:
# # plot sal and S_2 to see despiked data vs. original 
# # make plot of S2 and sal (i.e before and after despiking)
# years = df.query("mooring == 'D' & instrument == 'ctd' & position == 'mid1'")['year'].unique()
# fig = plt.figure(figsize=(12,4))
# # plot_timeseries(years=years,var='S_0',mooring='D',instrument='ctd',position='top',color='tab:blue',label='S_0')
# # plot_timeseries(years=years,var='S_1',mooring='D',instrument='ctd',position='top',color='tab:green',label='S_1')
# plot_timeseries(years=years,var='S_2',mooring='D',instrument='ctd',position='mid1',color='tab:red',label='S_2')
# plot_timeseries(years=years,var='sal',mooring='D',instrument='ctd',position='mid1',color='tab:blue',label='sal')

# years = df.query("mooring == 'D' & instrument == 'ctd' & position == 'mid2'")['year'].unique()
# plot_timeseries(years=years,var='S_2',mooring='D',instrument='ctd',position='mid2',color='tab:red',label='S_2')
# plot_timeseries(years=years,var='sal',mooring='D',instrument='ctd',position='mid2',color='tab:blue',label='sal')

# # get handles and labels for legend 
# handles, labels = plt.gca().get_legend_handles_labels()
# # only picks unique labels / handles 
# unique = dict(zip(labels, handles))
# # makes legend with unique keys only 
# plt.legend(unique.values(), unique.keys())

# # set limits on plot 
# plt.ylim(34.2,34.95)
# plt.ylabel('Salinity')
# plt.suptitle('Salinity at Mooring D Mid1 Sensor')

#### Despike Mooring D Salinity Bottom Sensor 

In [ ]:
df.query("mooring == 'D' & position == 'bot' & instrument == 'ctd'")

##### 2014

In [ ]:
data, data_despiked_2, data_despiked_3, data_despiked_4, data_despiked_5, data_despiked_seasonal, flagged_times_seasonal = get_despiked_data(data = D_2014_ctd_bot.S_2)

In [ ]:
ax0, ax1, ax2, ax3, ax4 = plot_despiked_data()
# ax0.set_ylim(34.4,34.6)
# ax1.set_ylim(34.4,34.6)
# ax2.set_ylim(34.4,34.6)
# ax3.set_ylim(34.4,34.6)
# ax4.set_ylim(33.8,34.6)
plt.suptitle('Mooring D Bot S_2 2014')

In [ ]:
D_2014_ctd_bot['sal'] = ('time',data_despiked_5.data)

In [ ]:
# despiking via threshold of 5 
# dt['D']['2014']['ctd']['bot']['sal'] = data_despiked_5

Note that 2016 and 2017 bottom sensor does not have level 2 data!! Need to ask why and if level 0 data is good to use. Much less discrepency between level 0 and 2 at bottom sensor compared to top. 

#### 2016

In [ ]:
D_2016_ctd_bot

In [ ]:
data, data_despiked_2, data_despiked_3, data_despiked_4, data_despiked_5, data_despiked_seasonal, flagged_times_seasonal = get_despiked_data(data = D_2016_ctd_bot.S_0)

In [ ]:
ax0, ax1, ax2, ax3, ax4 = plot_despiked_data()
# ax0.set_ylim(34.4,34.6)
# ax1.set_ylim(34.4,34.6)
# ax2.set_ylim(34.4,34.6)
# ax3.set_ylim(34.4,34.6)
# ax4.set_ylim(33.8,34.6)
plt.suptitle('Mooring D Bot S_0 2016')

In [ ]:
outliers_2 = get_outliers(data=data,threshold=2)
outliers_3 = get_outliers(data=data,threshold=3)
outliers_4 = get_outliers(data=data,threshold=4)
outliers_5 = get_outliers(data=data,threshold=5)

flagged_times_2 = outliers_2.outlier_values.dropna(dim="time").time.values
flagged_times_3 = outliers_3.outlier_values.dropna(dim="time").time.values
flagged_times_4 = outliers_4.outlier_values.dropna(dim="time").time.values
flagged_times_5 = outliers_5.outlier_values.dropna(dim="time").time.values

In [ ]:
# need to crop start and end 
D_2016_ctd_bot = D_2016_ctd_bot.sel(time = slice('2016-02-06T22:20','2016-05-28T06:00'))



In [ ]:
time0 = np.datetime64('2016-03-26T00:32:00.000000000')
time1 = np.datetime64('2016-04-18T07:27:00.000000000')
time2 = np.datetime64('2016-05-20T11:57:00.000000000')
time3 = np.datetime64('2016-04-20T18:02:00.000000000')
time4 = np.datetime64('2016-05-10T22:07:00.000000000')
time5 = np.datetime64('2016-05-12T00:22:00.000000000')

In [ ]:
D_2016_ctd_bot['sal'] = ('time',D_2016_ctd_bot.S_0.data)

In [ ]:
D_2016_ctd_bot['sal'].loc[dict(time=time0)] = np.nan
D_2016_ctd_bot['sal'].loc[dict(time=time1)] = np.nan
D_2016_ctd_bot['sal'].loc[dict(time=time2)] = np.nan
D_2016_ctd_bot['sal'].loc[dict(time=time3)] = np.nan
D_2016_ctd_bot['sal'].loc[dict(time=time4)] = np.nan
D_2016_ctd_bot['sal'].loc[dict(time=time5)] = np.nan



In [ ]:
fig = plt.figure()
D_2016_ctd_bot.sal.plot()

#### 2017 

In [ ]:
data, data_despiked_2, data_despiked_3, data_despiked_4, data_despiked_5, data_despiked_seasonal, flagged_times_seasonal = get_despiked_data(data = D_2017_ctd_bot.S_0)

In [ ]:
ax0, ax1, ax2, ax3, ax4 = plot_despiked_data()
# ax0.set_ylim(34.4,34.6)
# ax1.set_ylim(34.4,34.6)
# ax2.set_ylim(34.4,34.6)
# ax3.set_ylim(34.4,34.6)
# ax4.set_ylim(33.8,34.6)
plt.suptitle('Mooring D Bot S_0 2017')

In [ ]:
time0 = np.datetime64('2017-03-18T14:35:00.000000000')

In [ ]:
D_2017_ctd_bot['sal'] = ('time', D_2017_ctd_bot.S_0.data)

In [ ]:
D_2017_ctd_bot['sal'].loc[dict(time=time0)] = np.nan

In [ ]:
fig = plt.figure()
D_2017_ctd_bot.sal.plot()

##### 2019

In [ ]:
data, data_despiked_2, data_despiked_3, data_despiked_4, data_despiked_5, data_despiked_seasonal, flagged_times_seasonal = get_despiked_data(data = D_2019_ctd_bot.S_2)

In [ ]:
ax0, ax1, ax2, ax3, ax4 = plot_despiked_data()
# ax0.set_ylim(34.4,34.6)
# ax1.set_ylim(34.4,34.6)
# ax2.set_ylim(34.4,34.6)
# ax3.set_ylim(34.4,34.6)
# ax4.set_ylim(33.8,34.6)
plt.suptitle('Mooring D Bot S_2 2019')

In [ ]:
time0 = np.datetime64('2019-07-05T17:30')

In [ ]:
time1 = np.datetime64('2019-05-08T10:00:00.000000000')

In [ ]:
D_2019_ctd_bot['sal'] = ('time', D_2019_ctd_bot.S_2.data)

In [ ]:
D_2019_ctd_bot['sal'].loc[dict(time=time0)] = np.nan
D_2019_ctd_bot['sal'].loc[dict(time=time1)] = np.nan

In [ ]:
# no despiking necessary for 2019 
# dt['D']['2019']['ctd']['bot']['sal'] = dt['D']['2019']['ctd']['bot']['S_2'].copy()

In [ ]:
fig = plt.figure()
D_2019_ctd_bot.sal.plot()

##### 2021

In [ ]:
data, data_despiked_2, data_despiked_3, data_despiked_4, data_despiked_5, data_despiked_seasonal, flagged_times_seasonal = get_despiked_data(data = D_2021_ctd_bot.S_2)

In [ ]:
ax0, ax1, ax2, ax3, ax4 = plot_despiked_data()
# ax0.set_ylim(34.4,34.6)
# ax1.set_ylim(34.4,34.6)
# ax2.set_ylim(34.4,34.6)
# ax3.set_ylim(34.4,34.6)
# ax4.set_ylim(33.8,34.6)
plt.suptitle('Mooring D Bot S_2 2021')

In [ ]:
D_2021_ctd_bot['sal'] = ('time', D_2021_ctd_bot.S_2.data)

In [ ]:
# no despiking necesary 
# dt['D']['2021']['ctd']['bot']['sal'] = dt['D']['2021']['ctd']['bot']['S_2'].copy()

##### 2023

In [ ]:
data, data_despiked_2, data_despiked_3, data_despiked_4, data_despiked_5, data_despiked_seasonal, flagged_times_seasonal = get_despiked_data(data = D_2023_ctd_bot.S_2)

In [ ]:
ax0, ax1, ax2, ax3, ax4 = plot_despiked_data()
# ax0.set_ylim(34.4,34.6)
# ax1.set_ylim(34.4,34.6)
# ax2.set_ylim(34.4,34.6)
# ax3.set_ylim(34.4,34.6)
# ax4.set_ylim(33.8,34.6)
plt.suptitle('Mooring D Bot S_2 2023')

In [ ]:
time0 = np.datetime64('2023-04-30T18:15:00.000000000')

In [ ]:
D_2023_ctd_bot['sal'] = ('time', D_2023_ctd_bot.S_2.data)

In [ ]:
D_2023_ctd_bot['sal'].loc[dict(time=time0)] = np.nan

In [ ]:
fig = plt.figure()
D_2023_ctd_bot.sal.plot()

In [ ]:
# crop first few points - first few points look like it could be real data 

In [ ]:
# not despiking 2023 - first spike may be an outlier but will keep for now 
# dt['D']['2023']['ctd']['bot']['sal'] = dt['D']['2023']['ctd']['bot']['S_2'].copy()

##### 2024

In [ ]:
data, data_despiked_2, data_despiked_3, data_despiked_4, data_despiked_5, data_despiked_seasonal, flagged_times_seasonal = get_despiked_data(data = D_2024_ctd_bot.S_2)

In [ ]:
ax0, ax1, ax2, ax3, ax4 = plot_despiked_data()
# ax0.set_ylim(34.4,34.6)
# ax1.set_ylim(34.4,34.6)
# ax2.set_ylim(34.4,34.6)
# ax3.set_ylim(34.4,34.6)
# ax4.set_ylim(33.8,34.6)
plt.suptitle('Mooring D Bot S_2 2024')

In [ ]:
D_2024_ctd_bot['sal'] = ('time', D_2024_ctd_bot.S_2.data)

In [ ]:
# not despiking 
# dt['D']['2024']['ctd']['bot']['sal'] = dt['D']['2024']['ctd']['bot']['S_2'].copy()

In [ ]:
# years = df.query("mooring == 'D' & instrument == 'ctd' & position == 'bot'")['year'].unique()
# fig = plt.figure(figsize=(12,4))
# # plot_timeseries(years=years,var='S_0',mooring='D',instrument='ctd',position='bot',color='tab:blue',label='S_0')
# # plot_timeseries(years=years,var='S_1',mooring='D',instrument='ctd',position='bot',color='tab:green',label='S_1')
# plot_timeseries(years=years,var='S_2',mooring='D',instrument='ctd',position='bot',color='tab:red',label='S_2')
# plot_timeseries(years=years,var='sal',mooring='D',instrument='ctd',position='bot',color='tab:blue',label='sal')

# # get handles and labels for legend 
# handles, labels = plt.gca().get_legend_handles_labels()
# # only picks unique labels / handles 
# unique = dict(zip(labels, handles))
# # makes legend with unique keys only 
# plt.legend(unique.values(), unique.keys())

# # set limits on plot 
# plt.ylim(34.6,34.95)
# plt.ylabel('Salinity')
# plt.suptitle('Salinity at Mooring D Bot Sensor')

Notes on salinity preprocessing: 

For mooring D, level S_2 data was used when availble. S_2 data was unavailabe for bottom sensor for the years 2016 and 2017, and top sensor for 2016. For these years, only level 0 salinity data is available. The top sensor showed more of a difference between level 2 and 0 data, while the bottom sensor did not show much of a difference. Perhaps it is ok to use level 0 salinity data for the bottom sensor, but not the top due to the larger discrepency. None of the level 2 data or level 0 data was despiked. I applied the modified zscore at various thresholds (2-5) for each time series, computed over the entire time series and seasonally, and viewed the effects of outlier detection on each time series to decide on which threshold to implement. This varied between timeseries likely due to the inherent variability. Only spikes that appeared as true spikes were removed, any data that could perhaps be a real signal (based on neighboring points, etc..) was kept. For each dataset in the datatree, a new variable named 'sal' was added, which is despiked level 2 data. This is the variable to use going forward for salinity. Note that mid1 sensor does not have salinity data until 2017 and in 2020 salinity data was sampled at mid2 position, not mid1, but depths are close enough in range and can likely be included together to represent mid depth 

#### Despike Temperature 

In [ ]:
data, data_despiked_2, data_despiked_3, data_despiked_4, data_despiked_5, data_despiked_seasonal, flagged_times_seasonal = get_despiked_data(data = D_2014_ctd_top.T_2)

In [ ]:
ax0, ax1, ax2, ax3, ax4 = plot_despiked_data()
# ax0.set_ylim(34.4,34.6)
# ax1.set_ylim(34.4,34.6)
# ax2.set_ylim(34.4,34.6)
# ax3.set_ylim(34.4,34.6)
# ax4.set_ylim(33.8,34.6)
plt.suptitle('Mooring D Top T_2 2014')

In the above plot, we can see that the modified zscore method of outlier detection does not work well on the temperature data. Too much real data is removed. Based on the preliminary plots in the beginning of the notebook, there are not a lot of concerning spikes in the level S_2 data. I will use the level S_2 data as is, and will rename to 'temp' in each dataset. Theta_0 variable has some concerning spikes in it. Level 2 temp data is not available for top sensor in 2016, and bottom sensor in 2016 and 2017. 

In [ ]:
# data table 
df.query("mooring == 'D' & position == 'top' & instrument == 'ctd'")

In [ ]:
# add temp variable to top ctd datasets 

D_2014_ctd_top['temp'] = ('time', D_2014_ctd_top.T_2.data)
D_2017_ctd_top['temp'] = ('time', D_2017_ctd_top.T_2.data)
D_2019_ctd_top['temp'] = ('time', D_2019_ctd_top.T_2.data)
D_2020_ctd_top['temp'] = ('time', D_2020_ctd_top.T_2.data)
D_2021_ctd_top['temp'] = ('time', D_2021_ctd_top.T_2.data)
D_2022_ctd_top['temp'] = ('time', D_2022_ctd_top.T_2.data)

In [ ]:
# years = df.query("mooring == 'D' & position == 'top' & instrument == 'ctd'")['year'].unique()

In [ ]:
# for i in range(len(years)):
#     var = 'T_2'
#     ds = dt['D'][years[i]]['ctd']['top']
#     if var in ds.data_vars:
#         ds['temp'] = ds['T_2']
#     else:
#         print('no')

In [ ]:
# years = df.query("mooring == 'D' & instrument == 'ctd' & position == 'top'")['year'].unique()
# fig = plt.figure(figsize=(12,4))
# # plot_timeseries(years=years,var='T_0',mooring='D',instrument='ctd',position='top',color='tab:blue',label='T_0')
# # plot_timeseries(years=years,var='Theta_0',mooring='D',instrument='ctd',position='top',color='tab:orange',label='Theta_0')
# plot_timeseries(years=years,var='temp',mooring='D',instrument='ctd',position='top',color='tab:green',label='temp')
# plot_timeseries(years=years,var='T_2',mooring='D',instrument='ctd',position='top',color='tab:red',label='T_2')

# # get handles and labels for legend 
# handles, labels = plt.gca().get_legend_handles_labels()
# # only picks unique labels / handles 
# unique = dict(zip(labels, handles))
# # makes legend with unique keys only 
# plt.legend(unique.values(), unique.keys())

# # set limits on plot 
# plt.ylim(-2.15,-1.8)
# plt.ylabel('Temp')
# plt.suptitle('Temp at Mooring D Top Sensor')

In [ ]:
df.query("mooring == 'D' & position == 'mid1' & instrument == 'ctd'")

In [ ]:
# add temp variable to mid1 datasets 
D_2017_ctd_mid1['temp'] = ('time',D_2017_ctd_mid1.T_2.data)
D_2019_ctd_mid1['temp'] = ('time',D_2019_ctd_mid1.T_2.data)
D_2021_ctd_mid1['temp'] = ('time',D_2021_ctd_mid1.T_2.data)
D_2022_ctd_mid1['temp'] = ('time',D_2022_ctd_mid1.T_2.data)
D_2023_ctd_mid1['temp'] = ('time',D_2023_ctd_mid1.T_2.data)
D_2024_ctd_mid1['temp'] = ('time',D_2024_ctd_mid1.T_2.data)

In [ ]:
# years = df.query("mooring == 'D' & position == 'mid1' & instrument == 'ctd'")['year'].unique()

In [ ]:
# for i in range(len(years)):
#     var = 'T_2'
#     ds = dt['D'][years[i]]['ctd']['mid1']
#     if var in ds.data_vars:
#         ds['temp'] = ds['T_2']
#     else:
#         print('no')

In [ ]:
df.query("mooring == 'D' & position == 'mid1' & instrument == 'temp'")

In [ ]:
# add temp to temp datasets above 
D_2014_temp_mid1['temp'] = ('time', D_2014_temp_mid1.T_2.data)
D_2016_temp_mid1['temp'] = ('time', D_2016_temp_mid1.T_2.data)
D_2020_temp_mid1['temp'] = ('time', D_2020_temp_mid1.T_2.data)

In [ ]:
# years = df.query("mooring == 'D' & position == 'mid1' & instrument == 'temp'")['year'].unique()

In [ ]:
# for i in range(len(years)):
#     var = 'T_2'
#     ds = dt['D'][years[i]]['temp']['mid1']
#     if var in ds.data_vars:
#         ds['temp'] = ds['T_2']
#     else:
#         print('no')

In [ ]:
years = df.query("mooring == 'D' & instrument == 'ctd' & position == 'mid1'")['year'].unique()
fig = plt.figure(figsize=(12,4))
# plot_timeseries(years=years,var='T_0',mooring='D',instrument='ctd',position='mid1',color='tab:blue',label='T_0')
# plot_timeseries(years=years,var='Theta_0',mooring='D',instrument='ctd',position='mid1',color='tab:orange',label='Theta_0')
plot_timeseries(years=years,var='temp',mooring='D',instrument='ctd',position='mid1',color='tab:green',label='temp')
plot_timeseries(years=years,var='T_2',mooring='D',instrument='ctd',position='mid1',color='tab:red',label='T_2')

# account for temp collected by thermistors, not ctds 
years = df.query("mooring == 'D' & instrument == 'temp' & position == 'mid1'")['year'].unique()
# plot_timeseries(years=years,var='T_0',mooring='D',instrument='temp',position='mid1',color='tab:blue',label='T_0')
# plot_timeseries(years=years,var='Theta_0',mooring='D',instrument='temp',position='mid1',color='tab:orange',label='Theta_0')
plot_timeseries(years=years,var='temp',mooring='D',instrument='temp',position='mid1',color='tab:green',label='temp')
plot_timeseries(years=years,var='T_2',mooring='D',instrument='temp',position='mid1',color='tab:red',label='T_2')

# get handles and labels for legend 
handles, labels = plt.gca().get_legend_handles_labels()
# only picks unique labels / handles 
unique = dict(zip(labels, handles))
# makes legend with unique keys only 
plt.legend(unique.values(), unique.keys())

# set limits on plot 
plt.ylim(-2,-1.85)
plt.ylabel('Temp')
plt.suptitle('Temp at Mooring D mid1 Sensor')

In [ ]:
df.query("mooring == 'D' & position == 'mid2' & instrument == 'temp'")

In [ ]:
# add temp data to mid2 temp datasets 
D_2014_temp_mid2['temp'] = ('time', D_2014_temp_mid2.T_2.data)
D_2016_temp_mid2['temp'] = ('time', D_2016_temp_mid2.T_2.data)
D_2017_temp_mid2['temp'] = ('time', D_2017_temp_mid2.T_2.data)
D_2019_temp_mid2['temp'] = ('time', D_2019_temp_mid2.T_2.data)
D_2021_temp_mid2['temp'] = ('time', D_2021_temp_mid2.T_2.data)
D_2022_temp_mid2['temp'] = ('time', D_2022_temp_mid2.T_2.data)
D_2023_temp_mid2['temp'] = ('time', D_2023_temp_mid2.T_2.data)
D_2024_temp_mid2['temp'] = ('time', D_2024_temp_mid2.T_2.data)


In [ ]:
# years = df.query("mooring == 'D' & position == 'mid2' & instrument == 'temp'")['year'].unique()

In [ ]:
# for i in range(len(years)):
#     var = 'T_2'
#     ds = dt['D'][years[i]]['temp']['mid2']
#     if var in ds.data_vars:
#         ds['temp'] = ds['T_2']
#     else:
#         print('no')

In [ ]:
df.query("mooring == 'D' & position == 'mid2' & instrument == 'ctd'")

In [ ]:
# add temp to mid2 ctd dataset 
D_2020_ctd_mid2['temp'] = ('time', D_2020_ctd_mid2.T_2.data)

In [ ]:
# years = df.query("mooring == 'D' & position == 'mid2' & instrument == 'ctd'")['year'].unique()

In [ ]:
# for i in range(len(years)):
#     var = 'T_2'
#     ds = dt['D'][years[i]]['ctd']['mid2']
#     if var in ds.data_vars:
#         ds['temp'] = ds['T_2']
#     else:
#         print('no')

In [ ]:
years = df.query("mooring == 'D' & instrument == 'ctd' & position == 'mid2'")['year'].unique()
fig = plt.figure(figsize=(12,4))
# plot_timeseries(years=years,var='T_0',mooring='D',instrument='ctd',position='mid2',color='tab:blue',label='T_0')
# plot_timeseries(years=years,var='Theta_0',mooring='D',instrument='ctd',position='mid2',color='tab:orange',label='Theta_0')
plot_timeseries(years=years,var='temp',mooring='D',instrument='ctd',position='mid2',color='tab:green',label='temp')
plot_timeseries(years=years,var='T_2',mooring='D',instrument='ctd',position='mid2',color='tab:red',label='T_2')

# account for temp collected by thermistors, not ctds 
years = df.query("mooring == 'D' & instrument == 'temp' & position == 'mid2'")['year'].unique()
# plot_timeseries(years=years,var='T_0',mooring='D',instrument='temp',position='mid2',color='tab:blue',label='T_0')
# plot_timeseries(years=years,var='Theta_0',mooring='D',instrument='temp',position='mid2',color='tab:orange',label='Theta_0')
plot_timeseries(years=years,var='temp',mooring='D',instrument='temp',position='mid2',color='tab:green',label='temp')
plot_timeseries(years=years,var='T_2',mooring='D',instrument='temp',position='mid2',color='tab:red',label='T_2')

# get handles and labels for legend 
handles, labels = plt.gca().get_legend_handles_labels()
# only picks unique labels / handles 
unique = dict(zip(labels, handles))
# makes legend with unique keys only 
plt.legend(unique.values(), unique.keys())

# set limits on plot 
plt.ylim(-2,-1.85)
plt.ylabel('Temp')
plt.suptitle('Temp at Mooring D mid2 Sensor')

In [ ]:
df.query("mooring == 'D' & position == 'bot' & instrument == 'ctd'")

In [ ]:
# add temp variable to bottom ctd datasets 
# note 2017 and 2016 do not have T_2 data - will use T_0 data for 2017, theta_0 for 2016 looks too off to use 
D_2014_ctd_bot['temp'] = ('time',D_2014_ctd_bot.T_2.data)
D_2017_ctd_bot['temp'] = ('time',D_2017_ctd_bot.T_0.data)
D_2019_ctd_bot['temp'] = ('time',D_2019_ctd_bot.T_2.data)
D_2021_ctd_bot['temp'] = ('time',D_2021_ctd_bot.T_2.data)
D_2023_ctd_bot['temp'] = ('time',D_2023_ctd_bot.T_2.data)
D_2024_ctd_bot['temp'] = ('time',D_2024_ctd_bot.T_2.data)


In [ ]:
# years = df.query("mooring == 'D' & position == 'bot' & instrument == 'ctd'")['year'].unique()

In [ ]:
# for i in range(len(years)):
#     var = 'T_2'
#     ds = dt['D'][years[i]]['ctd']['bot']
#     if var in ds.data_vars:
#         ds['temp'] = ds['T_2']
#     else:
#         print('no')

In [ ]:
years = df.query("mooring == 'D' & instrument == 'ctd' & position == 'bot'")['year'].unique()
fig = plt.figure(figsize=(12,4))
# plot_timeseries(years=years,var='T_0',mooring='D',instrument='ctd',position='bot',color='tab:blue',label='T_0')
# plot_timeseries(years=years,var='Theta_0',mooring='D',instrument='ctd',position='bot',color='tab:orange',label='Theta_0')
plot_timeseries(years=years,var='temp',mooring='D',instrument='ctd',position='bot',color='tab:green',label='temp')
plot_timeseries(years=years,var='T_2',mooring='D',instrument='ctd',position='bot',color='tab:red',label='T_2')

# get handles and labels for legend 
handles, labels = plt.gca().get_legend_handles_labels()
# only picks unique labels / handles 
unique = dict(zip(labels, handles))
# makes legend with unique keys only 
plt.legend(unique.values(), unique.keys())

# set limits on plot 
plt.ylim(-2,-1.85)
plt.ylabel('Temp')
plt.suptitle('Temp at Mooring D bot Sensor')

### Rename Variables in CTD and temp datasets 

In [ ]:
dataset_names

In [ ]:
import numpy as np
import xarray as xr

def make_depth_positive(ds, depth_name="z"):
    """
    Ensure depth coordinate is numeric and positive.

    - Converts string depth to float
    - Handles invalid values safely
    - Preserves attributes
    - Sets CF-compliant metadata

    Parameters
    ----------
    ds : xr.Dataset
    depth_name : str

    Returns
    -------
    xr.Dataset
    """

    if depth_name not in ds.coords:
        return ds

    z = ds[depth_name]

    # Convert to numeric (strings → float, invalid → NaN)
    z_numeric = xr.apply_ufunc(
        lambda x: np.asarray(x, dtype="float64"),
        z,
        dask="allowed",
        vectorize=True,
    )

    # Make positive
    z_pos = np.abs(z_numeric)

    # Preserve attributes
    z_pos.attrs = z.attrs.copy()
    z_pos.attrs.update({
        "standard_name": "depth",
        "long_name": "Depth (positive downward)",
        "positive": "down",
        "units": "m"
    })

    return ds.assign_coords({depth_name: z_pos})


In [ ]:
D_datasets = [make_depth_positive(ds) for ds in D_datasets]

In [ ]:
D_2014_ctd_bot

In [ ]:
# add in attributes 
# define matadata for coorindates and variables - this will be the same for top and bottom sensor 
META_DATA = {
    "pres": {
        "long_name": "Pressure",
        "units": "dbar",
    },
    "abs_sal": {
        "long_name": "Absolute Salinity",
        "units": "g/kg",
    },
    "cons_temp": {
        "long_name": "Conservative Temperature",
        "units": "degree_Celsius",
    },
    "cons_temp_freezing": {
        "long_name": "Conservative Temperature Freezing",
        "units": "degree_Celsius",
    },
    "sigma0": {
        "long_name": "Potential Density Anomaly",
        "units": "kg/m3",
    },
    "theta": {
        "long_name": "Potential Temperature",
        "units": "degree_Celsius",
    },
    "sal": {
        "long_name": "Practical salinity",
        "units": "PSU",
    },
    "temp_freezing": {
        "long_name": "Temperature Freezing [ITS-90 deg C]",
        "units": "degree_Celsius",
    },
    "temp": {
        "long_name": "Temperature [ITS-90 deg C]",
        "units": "degree_Celsius",
    },
    "con": {
        "long_name": "Conductivity",
        "units": "S/m",
    },
    "z": {
        "long_name": "Depth",
        "units": "m",
        "description": "Depth is positive downward"
    },
    "lat": {
        "long_name": "Latitude",
        "units": "degrees_north",
    },
    "lon": {
        "long_name": "Longitude",
        "units": "degrees_east",
    },
    "u":{
        "long_name": "Eastward Velocity",
        "units": "cm/s",
    },
    "v":{
        "long_name": "Northward Velocity",
        "units": "cm/s",
    },
    "w": {
        "long_name": "Vertical Velocity",
        "units": "m/s",
    },
    "speed": {
        "long_name": "speed",
        "units": "cm/s",
    },
    "dir": {
        "long_name": "direction",
        "units": "degrees",
    },
    "mag_declination": {
        "long_name": "Magnetic Declination",
        "units": "degrees East",
        "description": "Angle between magnetic North and true North"
    },
    
    
}

In [ ]:
for v, attrs in META_DATA.items():
    for i in range(len(D_datasets)):
        if v in D_datasets[i]:
            D_datasets[i][v].attrs.update(attrs)

In [ ]:
for i in range(len(D_datasets)):
    D_datasets[i].to_netcdf(path='//thepenguin/penguin2/Data/TNB/mooring/Mooring_D_MORSea/Mooring_D_2014_2025/working/'+dataset_names[i]+'.nc')

In [ ]:
## function for dropping variables - not doing this for now but may want to in the future 

# def drop_vars_if_exist(ds, vars_to_drop):
#     """
#     Drop variables from an xarray Dataset only if they exist.

#     Parameters
#     ----------
#     ds : xr.Dataset
#         Input dataset
#     vars_to_drop : list or iterable
#         Variable names to drop

#     Returns
#     -------
#     xr.Dataset
#         Dataset with variables dropped where applicable
#     """
#     existing = [v for v in vars_to_drop if v in ds.data_vars]
#     return ds.drop_vars(existing)


# vars_to_drop = [
#     'T_0',
#     'S_0',
#     'S_1',
#     'S_2',
#     'T_2',
#     'Theta_0'

# ]


### Preprocess Velocity Data

In [ ]:
def monthly_quiver(ds,title,xlabel,u,v):
    """ 
    """

    import matplotlib.dates as mdates
    fig, ax = plt.subplots(figsize=(12,4))
    q = plt.quiver(
        ds.time.resample(time='1M').mean().values,np.zeros_like(ds.time.resample(time='1M').mean().values, dtype=float),
        ds[u].resample(time='1M').mean(),
        ds[v].resample(time='1M').mean()
    )
    plt.quiverkey(q, X=0.90, Y=0.1, U=5, label='5 cm/s', labelpos='E')

    plt.ylabel('Velocity (m/s)')
    plt.ylim(-5,5)
    ax.xaxis.set_major_locator(mdates.MonthLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b'))

    # Show year once at the left
    ax.set_xlabel(xlabel)
    ax.set_title(title)

#### 2014

In [ ]:
# nothing to remove here 
fig = plt.figure(figsize=(12,8))

plt.subplot(2,1,1)
D_2014_vel_top.U_0.plot()

plt.subplot(2,1,2)
D_2014_vel_top.V_0.plot()

plt.suptitle('D 2014 Top')

In [ ]:
fig = plt.figure(figsize=(12,8))

plt.subplot(2,1,1)
D_2014_vel_mid.U_0.plot()

plt.subplot(2,1,2)
D_2014_vel_mid.V_0.plot()

plt.suptitle('D 2014 mid')

In [ ]:
# remove spike at beginning by cropping dataset 
D_2014_vel_mid = D_2014_vel_mid.sel(time=slice('2014-01-21T10:00',None))

In [ ]:
fig = plt.figure(figsize=(12,8))

plt.subplot(2,1,1)
D_2014_vel_bot.U_0.plot()

plt.subplot(2,1,2)
D_2014_vel_bot.V_0.plot()

plt.suptitle('D 2014 bot')

In [ ]:
# crop bottom sensor at beginning to remove large spike D
D_2014_vel_bot = D_2014_vel_bot.sel(time=slice('2014-01-21T09:00',None))

In [ ]:
monthly_quiver(ds=D_2024_vel_bot, title='D 2019 Bottom', xlabel='2019 Deployment', u='U_0', v='V_0')

#### 2016

In [ ]:
# what the fuck is this? 
fig = plt.figure(figsize=(12,8))

plt.subplot(2,1,1)
D_2016_vel_top.U_0.plot()

plt.subplot(2,1,2)
D_2016_vel_top.V_0.plot()

plt.suptitle('D 2016 top')

In [ ]:
# what the fuck is this? !!!! 
fig = plt.figure(figsize=(12,8))

plt.subplot(2,1,1)
D_2016_vel_mid.U_0.plot()

plt.subplot(2,1,2)
D_2016_vel_mid.V_0.plot()

plt.suptitle('D 2016 mid')

In [ ]:
# what the fuck is this? 
fig = plt.figure(figsize=(12,8))

plt.subplot(2,1,1)
D_2016_vel_bot.U_0.plot()

plt.subplot(2,1,2)
D_2016_vel_bot.V_0.plot()

plt.suptitle('D 2016_bot')

In [ ]:
# crop to remove first spike 
D_2016_vel_bot = D_2016_vel_bot.sel(time=slice('2016-02-06T23:00', None))

I will not use 2016 top and mid velocity data for now due to the many erroneous spikes. Need to find out if this is worth fixing. 

#### 2017

In [ ]:
# what the fuck is this? 
fig = plt.figure(figsize=(12,8))

plt.subplot(2,1,1)
D_2017_vel_top.U_0.plot()

plt.subplot(2,1,2)
D_2017_vel_top.V_0.plot()

plt.suptitle('D 2017 Top')

In [ ]:
# crop to remove first spike 
D_2017_vel_top = D_2017_vel_top.sel(time=slice('2017-02-11T20:00',None))

In [ ]:
# what the fuck is this? 
fig = plt.figure(figsize=(12,8))

plt.subplot(2,1,1)
D_2017_vel_mid.U_0.plot()

plt.subplot(2,1,2)
D_2017_vel_mid.V_0.plot()

plt.suptitle('D 2017 mid')

In [ ]:
# what the fuck is this? 
fig = plt.figure(figsize=(12,8))

plt.subplot(2,1,1)
D_2017_vel_bot.U_0.plot()

plt.subplot(2,1,2)
D_2017_vel_bot.V_0.plot()

plt.suptitle('D 2017 bot')

#### 2019

In [ ]:
# what the fuck is this? 
fig = plt.figure(figsize=(12,8))

plt.subplot(2,1,1)
D_2019_vel_top.U_0.plot()

plt.subplot(2,1,2)
D_2019_vel_top.V_0.plot()

plt.suptitle('D 2019 Top')

In [ ]:
# what the fuck is this? 
fig = plt.figure(figsize=(12,8))

plt.subplot(2,1,1)
D_2019_vel_mid.U_0.plot()

plt.subplot(2,1,2)
D_2019_vel_mid.V_0.plot()

plt.suptitle('D 2019 mid')

In [ ]:
# what the fuck is this? 
fig = plt.figure(figsize=(12,8))

plt.subplot(2,1,1)
D_2019_vel_bot.U_0.plot()

plt.subplot(2,1,2)
D_2019_vel_bot.V_0.plot()

plt.suptitle('D 2019 bot')

#### 2021 

In [ ]:
# what the fuck is this? 
fig = plt.figure(figsize=(12,8))

plt.subplot(2,1,1)
D_2021_vel_top.U_0.plot()

plt.subplot(2,1,2)
D_2021_vel_top.V_0.plot()

plt.suptitle('D 2021 Top')

In [ ]:
# what the fuck is this? 
fig = plt.figure(figsize=(12,8))

plt.subplot(2,1,1)
D_2021_vel_mid.U_0.plot()

plt.subplot(2,1,2)
D_2021_vel_mid.V_0.plot()

plt.suptitle('D 2021 mid')

In [ ]:
# what the fuck is this? 
fig = plt.figure(figsize=(12,8))

plt.subplot(2,1,1)
D_2021_vel_bot.U_0.plot()

plt.subplot(2,1,2)
D_2021_vel_bot.V_0.plot()

plt.suptitle('D 2021 bot')

#### 2022

In [ ]:
# what the fuck is this? 
fig = plt.figure(figsize=(12,8))

plt.subplot(2,1,1)
D_2022_vel_top.U_0.plot()

plt.subplot(2,1,2)
D_2022_vel_top.V_0.plot()

plt.suptitle('D 2022 Top')

In [ ]:
# what the fuck is this? 
fig = plt.figure(figsize=(12,8))

plt.subplot(2,1,1)
D_2022_vel_mid.U_0.plot()

plt.subplot(2,1,2)
D_2022_vel_mid.V_0.plot()

plt.suptitle('D 2022 mid')

No bottom velocity in 2022

#### 2023 

In [ ]:
# what the fuck is this? 
fig = plt.figure(figsize=(12,8))

plt.subplot(2,1,1)
D_2023_vel_mid.U_0.plot()

plt.subplot(2,1,2)
D_2023_vel_mid.V_0.plot()

plt.suptitle('D 2023 Mid')

In [ ]:
# what the fuck is this? 
fig = plt.figure(figsize=(12,8))

plt.subplot(2,1,1)
D_2023_vel_bot.U_0.plot()

plt.subplot(2,1,2)
D_2023_vel_bot.V_0.plot()

plt.suptitle('D 2023 bot')

#### 2024 

In [ ]:
# what the fuck is this? 
fig = plt.figure(figsize=(12,8))

plt.subplot(2,1,1)
D_2024_vel_mid.U_0.plot()

plt.subplot(2,1,2)
D_2024_vel_mid.V_0.plot()

plt.suptitle('D 2024 Mid')

In [ ]:
# what the fuck is this? 
fig = plt.figure(figsize=(12,8))

plt.subplot(2,1,1)
D_2024_vel_bot.U_0.plot()

plt.subplot(2,1,2)
D_2024_vel_bot.V_0.plot()

plt.suptitle('D 2024 bot')

In [ ]:
D_vel_datasets = [
    D_2014_vel_top,
    D_2014_vel_mid,
    D_2014_vel_bot,
    D_2016_vel_bot,
    D_2017_vel_top,
    D_2017_vel_mid,
    D_2017_vel_bot,
    D_2019_vel_top,
    D_2019_vel_mid,
    D_2019_vel_bot,
    D_2021_vel_top,
    D_2021_vel_mid,
    D_2021_vel_bot,
    D_2022_vel_top,
    D_2022_vel_mid,
    D_2023_vel_mid,
    D_2023_vel_bot,
    D_2024_vel_mid,
    D_2024_vel_bot
]

D_vel_datasets_names = [
    'D_2014_vel_top',
    'D_2014_vel_mid',
    'D_2014_vel_bot',
    'D_2016_vel_bot',
    'D_2017_vel_top',
    'D_2017_vel_mid',
    'D_2017_vel_bot',
    'D_2019_vel_top',
    'D_2019_vel_mid',
    'D_2019_vel_bot',
    'D_2021_vel_top',
    'D_2021_vel_mid',
    'D_2021_vel_bot',
    'D_2022_vel_top',
    'D_2022_vel_mid',
    'D_2023_vel_mid',
    'D_2023_vel_bot',
    'D_2024_vel_mid',
    'D_2024_vel_bot'
]

In [ ]:
import numpy as np
import xarray as xr

def make_depth_positive(ds, depth_name="z"):
    """
    Ensure depth coordinate is numeric and positive.

    - Converts string depth to float
    - Handles invalid values safely
    - Preserves attributes
    - Sets CF-compliant metadata

    Parameters
    ----------
    ds : xr.Dataset
    depth_name : str

    Returns
    -------
    xr.Dataset
    """

    if depth_name not in ds.coords:
        return ds

    z = ds[depth_name]

    # Convert to numeric (strings → float, invalid → NaN)
    z_numeric = xr.apply_ufunc(
        lambda x: np.asarray(x, dtype="float64"),
        z,
        dask="allowed",
        vectorize=True,
    )

    # Make positive
    z_pos = np.abs(z_numeric)

    # Preserve attributes
    z_pos.attrs = z.attrs.copy()
    z_pos.attrs.update({
        "standard_name": "depth",
        "long_name": "Depth (positive downward)",
        "positive": "down",
        "units": "m"
    })

    return ds.assign_coords({depth_name: z_pos})


In [ ]:
D_vel_datasets = [make_depth_positive(ds) for ds in D_vel_datasets]

In [ ]:
# reanme vars 

In [ ]:
# Create function for renaming dimensions 

def rename_vars_if_exist(ds, rename_map):
    """
    Rename variables in an xarray Dataset only if they exist.

    Parameters
    ----------
    ds : xr.Dataset
        Input dataset
    rename_map : dict
        Mapping {old_name: new_name}

    Returns
    -------
    xr.Dataset
        Dataset with variables renamed where applicable
    """
    existing = {old: new for old, new in rename_map.items() if old in ds.data_vars}
    return ds.rename(existing)


In [ ]:
D_2019_vel_bot

In [ ]:
rename_map = {
    "U_0": "u",
    "V_0": "v",

    
}

In [ ]:
# add in attributes 
# define matadata for coorindates and variables - this will be the same for top and bottom sensor 
META_DATA = {
    "pres": {
        "long_name": "Pressure",
        "units": "dbar",
    },
    "abs_sal": {
        "long_name": "Absolute Salinity",
        "units": "g/kg",
    },
    "cons_temp": {
        "long_name": "Conservative Temperature",
        "units": "degree_Celsius",
    },
    "cons_temp_freezing": {
        "long_name": "Conservative Temperature Freezing",
        "units": "degree_Celsius",
    },
    "sigma0": {
        "long_name": "Potential Density Anomaly",
        "units": "kg/m3",
    },
    "theta": {
        "long_name": "Potential Temperature",
        "units": "degree_Celsius",
    },
    "sal": {
        "long_name": "Practical salinity",
        "units": "PSU",
    },
    "temp_freezing": {
        "long_name": "Temperature Freezing [ITS-90 deg C]",
        "units": "degree_Celsius",
    },
    "temp": {
        "long_name": "Temperature [ITS-90 deg C]",
        "units": "degree_Celsius",
    },
    "con": {
        "long_name": "Conductivity",
        "units": "S/m",
    },
    "z": {
        "long_name": "Depth",
        "units": "m",
        "description": "Depth is positive downward"
    },
    "lat": {
        "long_name": "Latitude",
        "units": "degrees_north",
    },
    "lon": {
        "long_name": "Longitude",
        "units": "degrees_east",
    },
    "u":{
        "long_name": "Eastward Velocity",
        "units": "cm/s",
    },
    "v":{
        "long_name": "Northward Velocity",
        "units": "cm/s",
    },
    "w": {
        "long_name": "Vertical Velocity",
        "units": "m/s",
    },
    "speed": {
        "long_name": "speed",
        "units": "cm/s",
    },
    "dir": {
        "long_name": "direction",
        "units": "degrees",
    },
    "mag_declination": {
        "long_name": "Magnetic Declination",
        "units": "degrees East",
        "description": "Angle between magnetic North and true North"
    },
    
    
}

In [ ]:
for i in range(len(D_vel_datasets)):
    D_vel_datasets[i] = rename_vars_if_exist(D_vel_datasets[i], rename_map)

In [ ]:
for v, attrs in META_DATA.items():
    for i in range(len(D_vel_datasets)):
        if v in D_vel_datasets[i]:
            D_vel_datasets[i][v].attrs.update(attrs)

In [ ]:
D_vel_datasets[9]

In [ ]:
# save final datasets 
# for i in range(len(D_vel_datasets)):
#     D_vel_datasets[i].to_netcdf(path='//thepenguin/penguin2/Data/TNB/mooring/Mooring_D_MORSea/Mooring_D_2014_2025/working/'+D_vel_datasets_names[i]+'.nc')

In [ ]:
# rename vars 

### Create Merged Timeseries For Easy Analysis 

#### Salinity Top

In [ ]:
# salinity top 
df.query("mooring == 'D' & instrument == 'ctd' & position == 'top'")

In [ ]:
# not including 2016 for now because no S_2 level 
years = ['2014','2017','2019','2020','2021','2022']
ds_list = []
for i in range(len(years)):
    ds_list.append(dt['D'][years[i]]['ctd']['top']['sal'])

In [ ]:
D_top_sal = xr.concat(ds_list,dim='time')

In [ ]:
fig = plt.figure(figsize=(12,4))
plt.scatter(D_top_sal.time,D_top_sal,s=2,alpha=0.8)
plt.title('Mooring D Salinity Top Sensor')

#### Salinity Mid1 

In [ ]:
# salinity mid1 
df.query("mooring == 'D' & instrument == 'ctd' & position == 'mid1'")

In [ ]:
years = df.query("mooring == 'D' & instrument == 'ctd' & position == 'mid1'")['year'].unique()

In [ ]:
years

In [ ]:
ds_list = []
for i in range(len(years)):
    ds_list.append(dt['D'][years[i]]['ctd']['mid1']['sal'])

In [ ]:
D_mid1_sal = xr.concat(ds_list,dim='time')

In [ ]:
df.query("mooring == 'D' & instrument == 'ctd' & position == 'mid2'")

In [ ]:
D_mid2_sal = dt['D']['2020']['ctd']['mid2']['sal']

In [ ]:
D_mid1_sal = xr.concat([D_mid1_sal,D_mid2_sal],dim='time').sortby('time')

#### Salinity Bottom

In [ ]:
# salinity bot
df.query("mooring == 'D' & instrument == 'ctd' & position == 'bot'")

In [ ]:
years = ['2014','2019','2021','2023','2024']

In [ ]:
ds_list = []
for i in range(len(years)):
    ds_list.append(dt['D'][years[i]]['ctd']['bot']['sal'])

In [ ]:
D_bot_sal = xr.concat(ds_list,dim='time')

In [ ]:
fig = plt.figure(figsize=(12,4))
plt.scatter(D_top_sal.time,D_top_sal,s=0.5,alpha=0.8,label='Top')
plt.scatter(D_mid1_sal.time,D_mid1_sal,s=0.5,alpha=0.8,label='Mid')
plt.scatter(D_bot_sal.time,D_bot_sal,s=0.5,alpha=0.8,label='Bot')
plt.title('Mooring D Salinity')
plt.legend()
plt.ylabel('Salinity')
plt.xlabel('Time')

#### Temp Top 

In [ ]:
df.query("mooring == 'D' & instrument == 'ctd' & position == 'top'")

In [ ]:
# not including 2016 for now because no S_2 level 
years = ['2014','2017','2019','2020','2021','2022']
ds_list = []
for i in range(len(years)):
    ds_list.append(dt['D'][years[i]]['ctd']['top']['temp'])

In [ ]:
# FOR GLACIOLOGY FINAL PROJECT 
years = ['1998','1999','2000','2010','2012','2014','2017','2019','2020','2021','2022']
ds_list = []
for i in range(len(years)):
    ds_list.append(dt['D'][years[i]]['ctd']['top']['T_0'])

In [ ]:
D_top_temp = xr.concat(ds_list,dim='time')

In [ ]:
fig = plt.figure(figsize=(12,4))
plt.scatter(D_top_temp.time,D_top_temp,s=0.5,alpha=0.8)
# D_top_temp.plot.scatter(x='time',s=10,c='tab:blue')

In [ ]:
    D_top_temp['z'] = (
        D_top_temp['z']
        .astype(str)        # everything becomes a string
        .str.strip()        # remove spaces
        .astype(float)      # convert '-612.8' → -612.8
        .astype(int)        # convert -612.8 → -612
    )

In [ ]:
D_top_temp.to_netcdf('D_top_temp_1998_2024_GlaciologyFinal.nc')

#### Temp Mid1

In [ ]:
# temp mid1 
df.query("mooring == 'D' & instrument == 'temp' & position == 'mid1'")

In [ ]:
years = df.query("mooring == 'D' & instrument == 'ctd' & position == 'mid1'")['year'].unique()

In [ ]:
years

In [ ]:
ds_list = []
for i in range(len(years)):
    ds_list.append(dt['D'][years[i]]['ctd']['mid1']['temp'])

In [ ]:
# FOR GLACIOLOGY PROJECT 
ds_list = []
for i in range(len(years)):
    ds_list.append(dt['D'][years[i]]['ctd']['mid1']['T_0'])

In [ ]:
D_mid1_temp = xr.concat(ds_list,dim='time')

In [ ]:
df.query("mooring == 'D' & instrument == 'temp' & position == 'mid1'")

In [ ]:
# FOR GLACIOLOGY PROJECT 
## need to resample thalf an hour too be cooonnsiistent with ooother data
D_mid1_temp_2014 = dt['D']['2014']['temp']['mid1'].to_dataset().resample(time='30min').mean()
D_mid1_temp_2016 = dt['D']['2016']['temp']['mid1'].to_dataset().resample(time='30min').mean()

In [ ]:
## need to resample thalf an hour too be cooonnsiistent with ooother data
D_mid1_temp_2014 = dt['D']['2014']['temp']['mid1'].to_dataset().resample(time='30min').mean()
D_mid1_temp_2016 = dt['D']['2016']['temp']['mid1'].to_dataset().resample(time='30min').mean()

In [ ]:
D_mid1_temp_2014_2016 = xr.concat([D_mid1_temp_2014,D_mid1_temp_2016],dim='time')

In [ ]:
D_mid1_temp1 = xr.concat([D_mid1_temp_2014_2016.T_0,dt['D']['2020']['temp']['mid1']['T_0']],dim='time').sortby('time')

In [ ]:
D_mid1_temp = xr.concat([D_mid1_temp,D_mid1_temp1],dim='time').sortby('time')

In [ ]:
fig = plt.figure(figsize=(12,4))
plt.scatter(D_mid1_temp.time,D_mid1_temp,s=1)
plt.ylim(-2.5,-1.8)

#### Temp Mid2

In [ ]:
df.query("mooring == 'D' & instrument == 'temp' & position == 'mid2'")

In [ ]:
# resample 2014 and 2016 to 30 min 
D_mid2_temp_2014 = dt['D']['2014']['temp']['mid2'].to_dataset().resample(time='30min').mean()
D_mid2_temp_2016 = dt['D']['2016']['temp']['mid2'].to_dataset().resample(time='30min').mean()

In [ ]:
D_mid2_temp_2014_2016 = xr.concat([D_mid2_temp_2014,D_mid2_temp_2016],dim='time')

In [ ]:
# temp mid2
years = df.query("mooring == 'D' & instrument == 'temp' & position == 'mid2'")['year'].unique()

In [ ]:
# exclude 2014 and 2016 
years = years[2:]

In [ ]:
ds_list = []
for i in range(len(years)):
    ds_list.append(dt['D'][years[i]]['temp']['mid2']['temp'])

In [ ]:
D_mid2_temp_2017_2024 = xr.concat(ds_list,dim='time')

In [ ]:
D_mid2_temp = xr.concat([D_mid2_temp_2014_2016.temp,D_mid2_temp_2017_2024],dim='time').sortby('time')

In [ ]:
df.query("mooring == 'D' & instrument == 'ctd' & position == 'mid2'")

In [ ]:
D_mid2_temp = xr.concat([D_mid2_temp,dt['D']['2020']['ctd']['mid2']['temp']],dim='time').sortby('time')

In [ ]:
# BOTTOM 
df.query("mooring == 'D' & instrument == 'ctd' & position == 'bot'")

In [ ]:
years = ['2014','2019','2021','2023','2024']
ds_list = []
for i in range(len(years)):
    ds_list.append(dt['D'][years[i]]['ctd']['bot']['temp'])

In [ ]:
D_bot_temp = xr.concat(ds_list,dim='time')

In [ ]:
fig = plt.figure(figsize=(12,4))
plt.scatter(D_top_temp.time,D_top_temp,s=0.5,alpha=0.8,label='Top')
plt.scatter(D_mid1_temp.time,D_mid1_temp,s=0.5,alpha=0.8,label='Mid1')
plt.scatter(D_mid2_temp.time,D_mid2_temp,s=0.5,alpha=0.8,label='Mid2')
plt.scatter(D_bot_temp.time,D_bot_temp,s=0.5,alpha=0.8,label='Bot')
plt.title('Mooring D Temperature')
plt.legend()
plt.ylabel('Temp')
plt.xlabel('Time')

In [ ]:
D_top_sal

In [ ]:
# depthh innn somme  iinstances is string - issues when saving needd to fix here first 
datasets = [D_top_temp, D_mid1_temp, D_mid2_temp, D_bot_temp, D_top_sal, D_mid1_sal, D_bot_sal]
dataset_names = ['D_top_temp', 'D_mid1_temp', 'D_mid2_temp', 'D_bot_temp', 'D_top_sal', 'D_mid1_sal', 'D_bot_sal']
for i,j in zip(range(len(datasets)),range(len(dataset_names))):
    ds = datasets[i]
    ds['z'] = (
        ds['z']
        .astype(str)        # everything becomes a string
        .str.strip()        # remove spaces
        .astype(float)      # convert '-612.8' → -612.8
        .astype(int)        # convert -612.8 → -612
    )
    ds.to_netcdf(f"{dataset_names[j]}.nc")


### T/S Diagrams 

In [ ]:
gsw.SA_from_SP?

In [ ]:
# gsw.sigma0 requires conservative temperature and absolute salinity 
# gsw.CT_from_t requires absolute salinity, in-situ temperature, and sea pressure 
# gsw.pt_from_t requires absolute salinity, in-situ temperature, sea pressure, and reference pressure 
# gsw.SA_from_SP requires practical salinity, sea pressure, longitude, latitude 
# Does the CTD measure practical salinity? 


In [ ]:
datasets = [D_top_temp, D_mid1_temp, D_mid2_temp, D_bot_temp, D_top_sal, D_mid1_sal, D_bot_sal]

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import gsw

# Extract variables
temp = D_bot_temp
sal  = D_bot_sal
time = pd.to_datetime(D_bot_sal.time.values)
years = time.year
unique_years = np.unique(years)

# ----- Density Grid -----
mint = temp.min()
maxt = temp.max()
mins = sal.min()
maxs = sal.max()

tempL = np.linspace(mint.values - 0.1, maxt.values + 0.1, 200)
salL  = np.linspace(mins.values - 0.1, maxs.values + 0.1, 200)
Tg, Sg = np.meshgrid(tempL, salL)

sigma_theta = gsw.sigma0(Sg, Tg)

# ----- Make Figure -----
fig, ax = plt.subplots(figsize=(8,6))

# Density contours
cs = ax.contour(Sg, Tg, sigma_theta, colors='grey', zorder=1, linewidths=0.8)
cl = plt.clabel(cs, fontsize=8, inline=False)

# ----- Scatter by Year -----
# cmap = plt.cm.get_cmap("tab20", len(unique_years))
cmap = plt.cm.get_cmap("jet", len(unique_years))

for i, yr in enumerate(unique_years):
    idx = (years == yr)
    ax.scatter(sal[idx], temp[idx], s=2, color=cmap(i),
               label=str(yr), zorder=3)

# ----- Freezing Temperature Line -----
t_freezing = gsw.t_freezing(salL, p=0, saturation_fraction=0)
ax.plot(salL, t_freezing, color='black', linewidth=1, zorder=2)

# ----- Labels + Decorations -----
ax.set_xlabel('Practical Salinity [g/kg]')
ax.set_ylabel('Potential Temperature [$^\circ$C]')
ax.set_title('Mooring D T/S Bottom Sensor')

ax.legend(title="Year")
ax.grid(True, linestyle='--', alpha=0.3)
ax.tick_params(direction='out')

plt.tight_layout()
plt.show()


Need to redo T/S diagrams using using correct temp / salinitiy. Potential density contours are calculated using conservative temperature and absolute salinity. Need to calculate these quantities. 

### Correct for Magnetic Declination

In [ ]:
mag_dec_2014 = 134 
mag_dec_2016 = 134 
mag_dec_2017 = 134 
mag_dec_2019 = 134.5 
mag_dec_2020 = 133 
mag_dec_2021 = 133.43 
mag_dec_2022 = 133.31 
mag_dec_2023 = 133.6 
mag_dec_2024 = 133.72 



In [ ]:
true_dir = magnetic_declination.calculate_true_direction(D_2014_vel_top.Dir_0, mag_dec_2014)

In [ ]:
magnetic_declination_radians = np.deg2rad(mag_dec_2014)
u_prime, v_prime = magnetic_declination.rotate_velocity(D_2014_vel_top.U_0,D_2014_vel_top.V_0,-magnetic_declination_radians)

In [ ]:
new_dir = magnetic_declination.calculate_direction(u_prime,v_prime)

In [ ]:
fig = plt.figure(figsize=(12,4))
true_dir.plot.scatter(x='time')
new_dir.plot.scatter(x='time')


In [ ]:
magnetic_declination_radians = np.deg2rad(mag_dec_2014)
u_prime, v_prime = magnetic_declination.rotate_velocity(D_2014_vel_top.U_0,D_2014_vel_top.V_0,-magnetic_declination_radians)
D_2014_vel_top['u'] = u_prime
D_2014_vel_top['v'] = v_prime
D_2014_vel_top['dir'] = new_dir 

### Get other data 

In [ ]:
df.query("mooring == 'DITN' & position == 'mid' & instrument == 'ctd'")

In [ ]:
# lowest temporal frequency is 10 minutes so will resample to that 
years = ['2017','2018','2019','2020']

In [ ]:
DITN_2017_ctd_mid = dt['DITN']['2017']['ctd']['mid'].to_dataset().resample(time='10min').mean()

In [ ]:
ds_list = [dt['DITN']['2014']['ctd']['mid'].to_dataset(),dt['DITN']['2015']['ctd']['mid'].to_dataset(),DITN_2017_ctd_mid] 

In [ ]:
DITN_glaciology_final = xr.concat(ds_list,dim='time')

In [ ]:
fig = plt.figure(figsize=(12,4))
DITN_glaciology_final.T.plot()

In [ ]:
df.query("mooring == 'DITS' ")

In [ ]:
fig = plt.figure(figsize=(12,4))
dt['DITS']['2017']['temp']['bot'].to_dataset().T.plot()

In [ ]:
df.query("mooring == 'LDEO' ")

In [ ]:
DITN_glaciology_final.to_netcdf('DITN_glaciology_final.nc')